In [1]:
!git clone https://github.com/guijiejie/DCMD-main.git

Cloning into 'DCMD-main'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 74 (delta 22), reused 48 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (74/74), 262.60 KiB | 4.38 MiB/s, done.
Resolving deltas: 100% (22/22), done.


In [2]:
# Clone SkateFormer repository 03/10/2025
!git clone https://github.com/KAIST-VICLab/SkateFormer.git
!git clone https://github.com/guijiejie/DCMD-main.git

# Check structure
!ls -la /kaggle/working/

Cloning into 'SkateFormer'...
remote: Enumerating objects: 170, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 170 (delta 14), reused 1 (delta 1), pack-reused 155 (from 2)
Receiving objects: 100% (170/170), 1.10 MiB | 7.19 MiB/s, done.
Resolving deltas: 100% (76/76), done.
fatal: destination path 'DCMD-main' already exists and is not an empty directory.
total 276
drwxr-xr-x 4 root root   4096 Oct  3 10:04 .
drwxr-xr-x 8 root root   4096 Oct  3 10:04 ..
drwxr-xr-x 7 root root   4096 Oct  3 10:04 DCMD-main
---------- 1 root root 264469 Oct  3 10:04 __notebook__.ipynb
drwxr-xr-x 9 root root   4096 Oct  3 10:04 SkateFormer


In [3]:
# Install required packages
!pip install -q appdirs==1.4.4 docker-pycreds==0.4.0 gitdb==4.0.10 gitpython==3.1.32 \
    joblib==1.3.1 numpy==1.25.2 pathtools==0.1.2 protobuf==4.23.4 \
    scikit-learn==1.3.0 scipy==1.11.1 sentry-sdk==1.29.2 setproctitle==1.3.2 \
    smmap==5.0.0 threadpoolctl==3.2.0 wandb==0.15.8

!pip install -q --upgrade pytorch-lightning pyyaml wandb pandas matplotlib \
    matplotlib-inline tqdm einops timm

!pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

print("✅ All packages installed successfully!")

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.5/188.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.5/304.5 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.2/36.2 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.6/215.6 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 28.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency

In [4]:
import os

# Create necessary directories
dirs = [
    '/kaggle/working/models',
    '/kaggle/working/config/Avenue',
    '/kaggle/working/checkpoints',
    '/kaggle/working/checkpoints/HR-Avenue',
    '/kaggle/working/checkpoints/HR-Avenue/skateformer_train',
    '/kaggle/working/checkpoints/HR-Avenue/skateformer_test',
]

for dir_path in dirs:
    os.makedirs(dir_path, exist_ok=True)
    print(f"✅ Created: {dir_path}")

✅ Created: /kaggle/working/models
✅ Created: /kaggle/working/config/Avenue
✅ Created: /kaggle/working/checkpoints
✅ Created: /kaggle/working/checkpoints/HR-Avenue
✅ Created: /kaggle/working/checkpoints/HR-Avenue/skateformer_train
✅ Created: /kaggle/working/checkpoints/HR-Avenue/skateformer_test


# **SkateFormer Core Model - Part 1 (Attention)**

In [5]:
%%writefile /kaggle/working/models/skateformer_attention.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Tuple

class PartitionAttention(nn.Module):
    """
    SkateFormer Partition-based Self-Attention
    Chia joints thành các partitions để giảm complexity
    """
    def __init__(self, dim, num_heads=8, num_joints=17, qkv_bias=False, 
                 attn_drop=0., proj_drop=0., partition_type='spatial'):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.num_joints = num_joints
        self.partition_type = partition_type
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        """
        x: [B, T*V, C] hoặc [B, N, C] - flattened temporal-spatial
        """
        B, N, C = x.shape
        
        # QKV projection
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # [3, B, num_heads, N, C//num_heads]
        q, k, v = qkv[0], qkv[1], qkv[2]

        # Attention
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        # Aggregate
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        
        return x


class SpatialAttention(nn.Module):
    """Spatial attention across joints"""
    def __init__(self, dim, num_heads=8, num_joints=17, qkv_bias=False, 
                 attn_drop=0., proj_drop=0.):
        super().__init__()
        self.attn = PartitionAttention(
            dim, num_heads, num_joints, qkv_bias, 
            attn_drop, proj_drop, partition_type='spatial'
        )
        
    def forward(self, x):
        """
        x: [B, C, T, V] -> process each frame independently
        """
        B, C, T, V = x.shape
        # [B, C, T, V] -> [B*T, V, C]
        x = x.permute(0, 2, 3, 1).contiguous().view(B*T, V, C)
        x = self.attn(x)
        # [B*T, V, C] -> [B, C, T, V]
        x = x.view(B, T, V, C).permute(0, 3, 1, 2).contiguous()
        return x


class TemporalAttention(nn.Module):
    """Temporal attention across frames"""
    def __init__(self, dim, num_heads=8, num_joints=17, qkv_bias=False, 
                 attn_drop=0., proj_drop=0.):
        super().__init__()
        self.attn = PartitionAttention(
            dim, num_heads, num_joints, qkv_bias, 
            attn_drop, proj_drop, partition_type='temporal'
        )
        
    def forward(self, x):
        """
        x: [B, C, T, V] -> process each joint independently
        """
        B, C, T, V = x.shape
        # [B, C, T, V] -> [B*V, T, C]
        x = x.permute(0, 3, 2, 1).contiguous().view(B*V, T, C)
        x = self.attn(x)
        # [B*V, T, C] -> [B, C, T, V]
        x = x.view(B, V, T, C).permute(0, 3, 2, 1).contiguous()
        return x

print("✅ SkateFormer Attention Module created!")

Writing /kaggle/working/models/skateformer_attention.py


# SkateFormer Core Model - Part 2 (Transformer Block)

In [6]:
%%writefile /kaggle/working/models/skateformer_block.py
import torch
import torch.nn as nn
import sys
sys.path.append('/kaggle/working/models')
from skateformer_attention import SpatialAttention, TemporalAttention


class Mlp(nn.Module):
    """MLP module"""
    def __init__(self, in_features, hidden_features=None, out_features=None, 
                 act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class SkateFormerBlock(nn.Module):
    """
    SkateFormer Transformer Block
    Spatial-Temporal attention với residual connections
    """
    def __init__(self, dim, num_heads, num_joints=17, mlp_ratio=4., 
                 qkv_bias=False, drop=0., attn_drop=0., 
                 attention_type='spatial'):
        super().__init__()
        self.attention_type = attention_type
        self.norm1 = nn.LayerNorm(dim)
        
        # Choose attention type
        if attention_type == 'spatial':
            self.attn = SpatialAttention(
                dim, num_heads=num_heads, num_joints=num_joints,
                qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop
            )
        elif attention_type == 'temporal':
            self.attn = TemporalAttention(
                dim, num_heads=num_heads, num_joints=num_joints,
                qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop
            )
        else:
            raise ValueError(f"Unknown attention type: {attention_type}")
        
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, 
                       act_layer=nn.GELU, drop=drop)

    def forward(self, x):
        """
        x: [B, C, T, V]
        """
        B, C, T, V = x.shape
        
        # Attention with residual
        identity = x
        
        # Reshape for LayerNorm: [B, C, T, V] -> [B, T, V, C]
        x = x.permute(0, 2, 3, 1).contiguous()
        x = self.norm1(x)
        # Back to [B, C, T, V]
        x = x.permute(0, 3, 1, 2).contiguous()
        
        x = self.attn(x)
        x = identity + x
        
        # MLP with residual
        identity = x
        # Reshape for LayerNorm and MLP
        x = x.permute(0, 2, 3, 1).contiguous()  # [B, T, V, C]
        x = self.norm2(x)
        x = self.mlp(x)
        x = x.permute(0, 3, 1, 2).contiguous()  # [B, C, T, V]
        x = identity + x
        
        return x


class STBlock(nn.Module):
    """
    Spatial-Temporal Block
    Kết hợp spatial và temporal attention
    """
    def __init__(self, dim, num_heads, num_joints=17, mlp_ratio=4., 
                 qkv_bias=False, drop=0., attn_drop=0.):
        super().__init__()
        
        # Spatial block
        self.spatial_block = SkateFormerBlock(
            dim, num_heads, num_joints, mlp_ratio, 
            qkv_bias, drop, attn_drop, attention_type='spatial'
        )
        
        # Temporal block
        self.temporal_block = SkateFormerBlock(
            dim, num_heads, num_joints, mlp_ratio, 
            qkv_bias, drop, attn_drop, attention_type='temporal'
        )
    
    def forward(self, x):
        """
        x: [B, C, T, V]
        """
        # Spatial then temporal
        x = self.spatial_block(x)
        x = self.temporal_block(x)
        return x

print("✅ SkateFormer Block Module created!")

Writing /kaggle/working/models/skateformer_block.py


#  SkateFormer Encoder (Thay thế STSAE)

In [7]:
%%writefile /kaggle/working/models/skateformer_encoder.py
import torch
import torch.nn as nn
import math
import sys
sys.path.append('/kaggle/working/models')
from skateformer_block import STBlock


class SkateFormerEncoder(nn.Module):
    """
    SkateFormer Encoder thay thế STSAE trong DCMD
    Input: [B, C, T, V] - Batch, Coords, Time, Joints
    Output: latent [B, 256], reconstruction [B, C, T, V]
    """
    def __init__(self, num_coords=2, num_joints=17, n_frames=3, 
                 embed_dim=256, depth=3, num_heads=8, mlp_ratio=4.,
                 drop_rate=0., attn_drop_rate=0.):
        super().__init__()
        self.num_coords = num_coords
        self.num_joints = num_joints
        self.n_frames = n_frames
        self.embed_dim = embed_dim
        
        # Input embedding: map coords to embed_dim
        self.input_embed = nn.Conv2d(num_coords, embed_dim, 
                                     kernel_size=1, stride=1)
        
        # Positional embedding
        self.pos_embed_temporal = nn.Parameter(
            torch.zeros(1, embed_dim, n_frames, 1)
        )
        self.pos_embed_spatial = nn.Parameter(
            torch.zeros(1, embed_dim, 1, num_joints)
        )
        
        # Spatial-Temporal blocks
        self.blocks = nn.ModuleList([
            STBlock(
                dim=embed_dim, num_heads=num_heads, num_joints=num_joints,
                mlp_ratio=mlp_ratio, qkv_bias=True, drop=drop_rate,
                attn_drop=attn_drop_rate
            )
            for _ in range(depth)
        ])
        
        # Global pooling and latent projection
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.latent_proj = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
        )
        
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(256, embed_dim),
            nn.ReLU(),
        )
        self.output_proj = nn.Conv2d(embed_dim, num_coords, 
                                     kernel_size=1, stride=1)
        
        # Initialize
        nn.init.trunc_normal_(self.pos_embed_temporal, std=0.02)
        nn.init.trunc_normal_(self.pos_embed_spatial, std=0.02)

    def forward(self, x):
        """
        Args:
            x: [B, C, T, V] - Input pose sequence
        Returns:
            latent: [B, 256] - Latent representation
            recon: [B, C, T, V] - Reconstructed poses
        """
        B, C, T, V = x.shape
        
        # Input embedding
        x = self.input_embed(x)  # [B, embed_dim, T, V]
        
        # Add positional embeddings
        x = x + self.pos_embed_temporal + self.pos_embed_spatial
        
        # Apply ST blocks
        for block in self.blocks:
            x = block(x)
        
        # Extract latent (global pooling)
        latent_feat = self.global_pool(x).squeeze(-1).squeeze(-1)  # [B, embed_dim]
        latent = self.latent_proj(latent_feat)  # [B, 256]
        
        # Decode for reconstruction
        dec_feat = self.decoder(latent)  # [B, embed_dim]
        dec_feat = dec_feat.unsqueeze(-1).unsqueeze(-1)  # [B, embed_dim, 1, 1]
        dec_feat = dec_feat.expand(-1, -1, T, V)  # [B, embed_dim, T, V]
        
        # Add skip connection from encoder
        dec_feat = dec_feat + x
        
        # Output projection
        recon = self.output_proj(dec_feat)  # [B, C, T, V]
        
        return latent, recon

print("✅ SkateFormer Encoder created!")

Writing /kaggle/working/models/skateformer_encoder.py


# SkateFormer Diffusion Model (Thay thế MotionTransformer)

In [8]:
%%writefile /kaggle/working/models/skateformer_diffusion.py
import torch
import torch.nn as nn
import math
import sys
sys.path.append('/kaggle/working/models')
from skateformer_block import STBlock


def timestep_embedding(timesteps, dim, max_period=10000):
    """Sinusoidal timestep embeddings"""
    half = dim // 2
    freqs = torch.exp(
        -math.log(max_period) * torch.arange(start=0, end=half, dtype=torch.float32) / half
    ).to(device=timesteps.device)
    args = timesteps[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding


class SkateFormerDiffusion(nn.Module):
    """
    SkateFormer-based Diffusion Model cho motion prediction
    Thay thế MotionTransformer trong DCMD
    """
    def __init__(self, input_feats, num_frames=7, latent_dim=512,
                 depth=4, num_heads=8, num_joints=17, dropout=0.1,
                 inject_condition=True):
        super().__init__()
        
        self.input_feats = input_feats  # 2 * num_joints = 34
        self.num_frames = num_frames
        self.latent_dim = latent_dim
        self.num_joints = num_joints
        self.num_coords = 2
        self.inject_condition = inject_condition
        
        # Time embedding
        self.time_embed_dim = latent_dim
        self.time_embed = nn.Sequential(
            nn.Linear(latent_dim, self.time_embed_dim),
            nn.SiLU(),
            nn.Linear(self.time_embed_dim, self.time_embed_dim),
        )
        
        # Condition embedding
        if inject_condition:
            self.cond_embed = nn.Linear(256, self.time_embed_dim)
        
        # Input projection: [B, T, input_feats] -> [B, latent_dim, T, V]
        self.input_proj = nn.Linear(input_feats, latent_dim)
        
        # Reshape to spatial-temporal format
        # We need to convert [B, T, input_feats] -> [B, C, T, V]
        # where input_feats = C * V
        
        # ST blocks
        self.blocks = nn.ModuleList([
            STBlock(
                dim=latent_dim, num_heads=num_heads, num_joints=num_joints,
                mlp_ratio=4., qkv_bias=True, drop=dropout,
                attn_drop=dropout
            )
            for _ in range(depth)
        ])
        
        # Time embedding injection layers
        self.time_inject = nn.ModuleList([
            nn.Sequential(
                nn.SiLU(),
                nn.Linear(self.time_embed_dim, latent_dim),
            )
            for _ in range(depth)
        ])
        
        # Output projection
        self.out_proj = nn.Linear(latent_dim, input_feats)
        
        # Initialize
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x, timesteps, condition_data=None):
        """
        Args:
            x: [B, T, D] where D = input_feats = C*V
            timesteps: [B]
            condition_data: [B, 256]
        Returns:
            output: [B, T, D]
        """
        B, T, D = x.shape
        
        # Time embedding
        t_emb = self.time_embed(timestep_embedding(timesteps, self.latent_dim))
        
        # Condition embedding
        if self.inject_condition and condition_data is not None:
            cond_emb = self.cond_embed(condition_data)
            emb = t_emb + cond_emb  # [B, latent_dim]
        else:
            emb = t_emb
        
        # Input projection: [B, T, D] -> [B, T, latent_dim]
        h = self.input_proj(x)
        
        # Reshape to [B, C, T, V] format for ST blocks
        # D = num_coords * num_joints, latent_dim plays role of channels
        h = h.permute(0, 2, 1).contiguous()  # [B, latent_dim, T]
        h = h.unsqueeze(-1).expand(-1, -1, -1, self.num_joints)  # [B, latent_dim, T, V]
        
        # Apply ST blocks with time embedding injection
        for i, (block, time_inj) in enumerate(zip(self.blocks, self.time_inject)):
            # Inject time+condition embedding
            t_feat = time_inj(emb)  # [B, latent_dim]
            t_feat = t_feat.unsqueeze(-1).unsqueeze(-1)  # [B, latent_dim, 1, 1]
            h = h + t_feat
            
            # ST block
            h = block(h)
        
        # Back to [B, T, D] format
        h = h.mean(dim=-1)  # [B, latent_dim, T] - average over joints
        h = h.permute(0, 2, 1).contiguous()  # [B, T, latent_dim]
        
        # Output projection
        output = self.out_proj(h)  # [B, T, D]
        
        # Return format compatible with DCMD
        return output, None, None, None

print("✅ SkateFormer Diffusion Model created!")

Writing /kaggle/working/models/skateformer_diffusion.py


# Copy utils từ DCMD

In [9]:
# Copy utility files from DCMD
import shutil
import os

# Source directory
dcmd_dir = '/kaggle/working/DCMD-main'

# Copy utils
utils_dir = os.path.join(dcmd_dir, 'utils')
if os.path.exists(utils_dir):
    shutil.copytree(utils_dir, '/kaggle/working/utils', dirs_exist_ok=True)
    print("✅ Copied utils from DCMD")

# Copy models/stsae if needed
stsae_dir = os.path.join(dcmd_dir, 'models/stsae')
if os.path.exists(stsae_dir):
    os.makedirs('/kaggle/working/models/stsae', exist_ok=True)
    shutil.copytree(stsae_dir, '/kaggle/working/models/stsae', dirs_exist_ok=True)
    print("✅ Copied STSAE models")

print("✅ All utils copied!")

✅ Copied utils from DCMD
✅ Copied STSAE models
✅ All utils copied!


#  DCMD + SkateFormer Integration (Main Model) - Part 1

In [10]:
%%writefile /kaggle/working/models/dcmd_skateformer.py
import argparse
import os
from math import prod
from typing import Dict, List, Tuple
import numpy as np
import pytorch_lightning as pl
import torch
import torch.nn as nn
from torch.optim import Adam
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

# Import SkateFormer components
import sys
sys.path.append('/kaggle/working')
sys.path.append('/kaggle/working/models')
sys.path.append('/kaggle/working/DCMD-main')

from models.skateformer_encoder import SkateFormerEncoder
from models.skateformer_diffusion import SkateFormerDiffusion
from utils.diffusion_utils import Diffusion
from utils.eval_utils import (compute_var_matrix, filter_vectors_by_cond,
                              get_avenue_mask, get_hr_ubnormal_mask, pad_scores, score_process)
from utils.model_utils import processing_data
from utils.tools import get_dct_matrix, generate_pad, padding_traj


class DCMD_SkateFormer(pl.LightningModule):
    """
    DCMD với SkateFormer Architecture
    
    Improvements:
    1. Partition-based attention từ SkateFormer
    2. Spatial-temporal modeling tốt hơn
    3. Efficient training với reduced complexity
    """
    
    losses = {'l1': nn.L1Loss, 'smooth_l1': nn.SmoothL1Loss, 'mse': nn.MSELoss}
    
    def __init__(self, args: argparse.Namespace) -> None:
        super().__init__()
        
        self.save_hyperparameters(args)
        
        # Data parameters
        self.n_frames = args.seg_len
        self.num_coords = args.num_coords
        self.n_joints = self._infer_number_of_joint(args)
        self.n_his = args.n_his
        
        # Model parameters
        self.dropout = args.dropout
        self.conditioning_strategy = 'inject'
        self.cond_latent_dim = args.latent_dim
        
        # Training parameters
        self.learning_rate = args.opt_lr
        self.loss_fn = self.losses[args.loss_fn](reduction='none')
        self.noise_steps = args.noise_steps
        
        # Inference parameters
        self.aggregation_strategy = args.aggregation_strategy
        self.n_generated_samples = args.n_generated_samples
        self.model_return_value = args.model_return_value
        
        # Paths and settings
        self.gt_path = args.gt_path
        self.split = args.split
        self.use_hr = args.use_hr
        self.ckpt_dir = args.ckpt_dir
        self.save_tensors = args.save_tensors
        self.num_transforms = args.num_transform
        self.anomaly_score_pad_size = args.pad_size
        self.anomaly_score_filter_kernel_size = args.filter_kernel_size
        self.anomaly_score_frames_shift = args.frames_shift
        self.dataset_name = args.dataset_choice
        self.padding = args.padding
        
        # Padding setup
        self.idx_pad, self.zero_index = generate_pad(
            self.padding, self.n_his, self.n_frames - self.n_his
        )
        
        # Diffusion setup
        self._set_diffusion_variables()
        
        # Build models
        self.build_model()
    
    def build_model(self) -> None:
        """Build SkateFormer-based models"""
        
        # SkateFormer Encoder (replaces STSAE)
        self.rec_model = SkateFormerEncoder(
            num_coords=self.num_coords,
            num_joints=self.n_joints,
            n_frames=self.n_his,
            embed_dim=256,
            depth=3,  # Reduced for faster training
            num_heads=8,
            mlp_ratio=4.,
            drop_rate=self.dropout,
            attn_drop_rate=self.dropout
        )
        
        # SkateFormer Diffusion (replaces MotionTransformer)
        self.pre_model = SkateFormerDiffusion(
            input_feats=self.num_coords * self.n_joints,
            num_frames=self.n_frames,
            latent_dim=512,
            depth=4,  # Reduced for faster training
            num_heads=8,
            num_joints=self.n_joints,
            dropout=self.dropout,
            inject_condition=True
        )
    
    def forward(self, input_data: List[torch.Tensor], 
                aggr_strategy: str = None, 
                return_: str = None) -> List[torch.Tensor]:
        """
        Forward pass
        
        Args:
            input_data: [tensor_data, transformation_idx, metadata, actual_frames]
            aggr_strategy: Aggregation strategy for multiple samples
            return_: What to return ('loss', 'poses', or 'all')
        
        Returns:
            List containing predictions and metadata
        """
        
        tensor_data, meta_out = self._unpack_data(input_data)
        B = tensor_data.shape[0]
        
        # Split into history and full sequence
        history_data = tensor_data[:, :, :self.n_his, :]
        x_0 = padding_traj(history_data, self.padding, self.idx_pad, self.zero_index)
        
        generated_xs = []
        
        # Generate multiple samples
        for _ in range(self.n_generated_samples):
            
            # 1. Reconstruction with SkateFormer Encoder
            condition_embedding, rec_his_data = self.rec_model(history_data)
            
            # 2. DCT transformation
            dct_m, idct_m = get_dct_matrix(self.n_frames)
            dct_m_all = dct_m.float().to(self.device)
            idct_m_all = idct_m.float().to(self.device)
            
            # Transform to DCT domain
            x = x_0.permute(0, 2, 3, 1).contiguous()  # [B, T, V, C]
            x = x.reshape([x.shape[0], self.n_frames, -1])  # [B, T, C*V]
            y = torch.matmul(dct_m_all, x)  # [B, T, C*V]
            
            # 3. Diffusion process
            y_d = torch.randn_like(y, device=self.device)
            
            for i in reversed(range(1, self.noise_steps)):
                t = torch.full(size=(B,), fill_value=i, dtype=torch.long, device=self.device)
                t_prev = torch.full(size=(B,), fill_value=i, dtype=torch.long, device=self.device)
                t_prev[0] = 0
                
                noise_pre = torch.randn_like(y_d) if i > 1 else torch.zeros_like(y_d)
                
                # SkateFormer prediction
                predicted_noise_pre, _, _, _ = self.pre_model(
                    y_d, t, condition_data=condition_embedding
                )
                
                # Diffusion step
                alpha_pre = self._alpha[t][:, None, None]
                alpha_hat_pre = self._alpha_hat[t][:, None, None]
                beta_pre = self._beta[t][:, None, None]
                
                y_d = (1 / torch.sqrt(alpha_pre)) * \
                      (y_d - ((1 - alpha_pre) / (torch.sqrt(1 - alpha_hat_pre))) * predicted_noise_pre) + \
                      torch.sqrt(beta_pre) * noise_pre
                
                # Add noise to ground truth
                alpha_hat_prev = self._alpha_hat[t_prev][:, None, None]
                y_n = (torch.sqrt(alpha_hat_prev) * y) + (torch.sqrt(1 - alpha_hat_prev) * noise_pre)
                
                # Mask completion (history frames are known)
                mask = torch.zeros_like(x, device=self.device)
                for m in range(0, self.n_his):
                    mask[:, m, :] = 1
                
                # iDCT for masking
                y_d_idct = torch.matmul(idct_m_all, y_d)
                y_n_idct = torch.matmul(idct_m_all, y_n)
                
                # Apply mask
                m_mul_y_n = torch.mul(mask, y_n_idct)
                m_mul_y_d = torch.mul((1 - mask), y_d_idct)
                m_y = m_mul_y_d + m_mul_y_n
                
                # DCT again
                y_d = torch.matmul(dct_m_all, m_y)
            
            # 4. iDCT to get final prediction
            pre_future_data = torch.matmul(idct_m_all, y_d)
            pre_future_data = pre_future_data.reshape(
                pre_future_data.shape[0], pre_future_data.shape[1], -1, 2
            )
            pre_future_data = pre_future_data.permute(0, 3, 1, 2).contiguous()
            
            # Select only future frames
            pre_future_data = pre_future_data[:, :, self.n_his:, :]
            
            # 5. Combine reconstruction + prediction
            xs = torch.cat((rec_his_data, pre_future_data), dim=2)
            generated_xs.append(xs)
        
        # Aggregate samples
        selected_x, loss_of_selected_x = self._aggregation_strategy(
            generated_xs, tensor_data, aggr_strategy
        )
        
        return self._pack_out_data(
            selected_x, loss_of_selected_x, [tensor_data] + meta_out, return_=return_
        )

print("✅ DCMD_SkateFormer Part 1 created!")

Writing /kaggle/working/models/dcmd_skateformer.py


# 10: DCMD + SkateFormer - Part 2 (Training)

In [11]:
%%writefile -a /kaggle/working/models/dcmd_skateformer.py

    def training_step(self, batch: List[torch.Tensor], batch_idx: int) -> torch.float32:
        """Training step - simplified without minimax"""
        
        tensor_data, _ = self._unpack_data(batch)
        
        history_data = tensor_data[:, :, :self.n_his, :]
        x_0 = tensor_data
        
        # 1. Reconstruction loss
        condition_embedding, rec_his_data = self.rec_model(history_data)
        rec_loss = torch.mean(self.loss_fn(rec_his_data, history_data))
        self.log('rec_loss', rec_loss, prog_bar=True)
        
        # 2. Prediction loss
        # DCT transformation
        dct_m, _ = get_dct_matrix(self.n_frames)
        dct_m_all = dct_m.float().to(self.device)
        x = x_0.permute(0, 2, 3, 1).contiguous()
        x = x.reshape([x.shape[0], self.n_frames, -1])
        y_0 = torch.matmul(dct_m_all, x)
        
        # Sample timesteps and add noise
        t = self.noise_scheduler.sample_timesteps(y_0.shape[0]).to(self.device)
        y_t, pre_noise = self.noise_scheduler.noise_motion(y_0, t)
        
        # Predict noise with SkateFormer
        pre_predicted_noise, _, _, _ = self.pre_model(
            y_t, t, condition_data=condition_embedding
        )
        
        # Prediction loss
        pre_loss = torch.mean(self.loss_fn(pre_predicted_noise, pre_noise))
        self.log('pre_loss', pre_loss, prog_bar=True)
        
        # 3. Total loss
        total_loss = rec_loss + pre_loss
        self.log('train_loss', total_loss, prog_bar=True)
        
        return total_loss
    
    def validation_step(self, batch: List[torch.Tensor], batch_idx: int) -> None:
        """Validation step"""
        self._validation_output_list.append(self.forward(batch))
    
    def on_validation_epoch_start(self) -> None:
        super().on_validation_epoch_start()
        self._validation_output_list = []
    
    def on_validation_epoch_end(self) -> float:
        """Validation epoch end"""
        if len(self._validation_output_list) == 0:
            return
        
        out, gt_data, trans, meta, frames = processing_data(self._validation_output_list)
        del self._validation_output_list
        
        auc_score = self.post_processing(out, gt_data, trans, meta, frames)
        self.log('val_AUC', auc_score, prog_bar=True)
        return auc_score
    
    def test_step(self, batch: List[torch.Tensor], batch_idx: int) -> None:
        """Test step"""
        self._test_output_list.append(self.forward(batch))
    
    def on_test_epoch_start(self) -> None:
        super().on_test_epoch_start()
        self._test_output_list = []
    
    def on_test_epoch_end(self) -> float:
        """Test epoch end"""
        out, gt_data, trans, meta, frames = processing_data(self._test_output_list)
        del self._test_output_list
        
        if self.save_tensors:
            tensors = {
                'prediction': out, 'gt_data': gt_data,
                'trans': trans, 'metadata': meta, 'frames': frames
            }
            self._save_tensors(
                tensors, split_name=self.split,
                aggr_strategy=self.aggregation_strategy,
                n_gen=self.n_generated_samples
            )
        
        auc_score = self.post_processing(out, gt_data, trans, meta, frames)
        self.log('test_AUC', auc_score, prog_bar=True)
        return auc_score
    
    def configure_optimizers(self) -> Dict:
        """Configure optimizer and scheduler"""
        optimizer = Adam(self.parameters(), lr=self.learning_rate, weight_decay=1e-5)
        
        # Cosine annealing scheduler
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=self.trainer.max_epochs, eta_min=1e-6
        )
        
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'interval': 'epoch',
                'monitor': 'train_loss'
            }
        }

print("✅ DCMD_SkateFormer Part 2 (Training) created!")

Appending to /kaggle/working/models/dcmd_skateformer.py


# DCMD + SkateFormer Integration (Main Model) - Part 2 (Forward)

In [12]:
%%writefile -a /kaggle/working/models/dcmd_skateformer.py

    def forward(self, input_data: List[torch.Tensor], aggr_strategy: str = None, 
                return_: str = None) -> List[torch.Tensor]:
        """Forward pass - tương tự DCMD nhưng dùng SkateFormer"""
        
        tensor_data, meta_out = self._unpack_data(input_data)
        B = tensor_data.shape[0]
        
        # Split history and full sequence
        history_data = tensor_data[:, :, :self.n_his, :]
        x_0 = padding_traj(history_data, self.padding, self.idx_pad, self.zero_index)
        
        generated_xs = []
        
        for _ in range(self.n_generated_samples):
            # 1. Reconstruction với SkateFormer Encoder
            condition_embedding, rec_his_data = self.rec_model(history_data)
            
            # 2. DCT transformation
            dct_m, idct_m = get_dct_matrix(self.n_frames)
            dct_m_all = dct_m.float().to(self.device)
            idct_m_all = idct_m.float().to(self.device)
            
            # [B, C, T, V] -> [B, T, V, C] -> [B, T, C*V]
            x = x_0.permute(0, 2, 3, 1).contiguous()
            x = x.reshape([x.shape[0], self.n_frames, -1])
            y = torch.matmul(dct_m_all, x)
            
            # 3. Diffusion process
            y_d = torch.randn_like(y, device=self.device)
            
            for i in reversed(range(1, self.noise_steps)):
                t = torch.full(size=(B,), fill_value=i, dtype=torch.long, device=self.device)
                t_prev = torch.full(size=(B,), fill_value=i, dtype=torch.long, device=self.device)
                t_prev[0] = 0
                
                noise_pre = torch.randn_like(y_d) if i > 1 else torch.zeros_like(y_d)
                
                # SkateFormer prediction
                predicted_noise_pre, _, _, _ = self.pre_model(
                    y_d, t, condition_data=condition_embedding
                )
                
                # Diffusion step
                alpha_pre = self._alpha[t][:, None, None]
                alpha_hat_pre = self._alpha_hat[t][:, None, None]
                beta_pre = self._beta[t][:, None, None]
                
                y_d = (1 / torch.sqrt(alpha_pre)) * \
                      (y_d - ((1 - alpha_pre) / (torch.sqrt(1 - alpha_hat_pre))) * predicted_noise_pre) + \
                      torch.sqrt(beta_pre) * noise_pre
                
                # Add noise to ground truth
                alpha_hat_prev = self._alpha_hat[t_prev][:, None, None]
                y_n = (torch.sqrt(alpha_hat_prev) * y) + (torch.sqrt(1 - alpha_hat_prev) * noise_pre)
                
                # Mask completion (keep history, predict future)
                mask = torch.zeros_like(x, device=self.device)
                for m in range(0, self.n_his):
                    mask[:, m, :] = 1
                
                y_d_idct = torch.matmul(idct_m_all, y_d)
                y_n_idct = torch.matmul(idct_m_all, y_n)
                m_mul_y_n = torch.mul(mask, y_n_idct)
                m_mul_y_d = torch.mul((1 - mask), y_d_idct)
                m_y = m_mul_y_d + m_mul_y_n
                y_d = torch.matmul(dct_m_all, m_y)
            
            # iDCT và reshape back
            pre_future_data = torch.matmul(idct_m_all, y_d)
            pre_future_data = pre_future_data.reshape(pre_future_data.shape[0], pre_future_data.shape[1], -1, 2)
            pre_future_data = pre_future_data.permute(0, 3, 1, 2).contiguous()
            pre_future_data = pre_future_data[:, :, self.n_his:, :]
            
            # Concatenate reconstruction + prediction
            xs = torch.cat((rec_his_data, pre_future_data), dim=2)
            generated_xs.append(xs)
        
        # Aggregate samples
        selected_x, loss_of_selected_x = self._aggregation_strategy(
            generated_xs, tensor_data, aggr_strategy
        )
        
        return self._pack_out_data(selected_x, loss_of_selected_x, 
                                   [tensor_data] + meta_out, return_=return_)

print("✅ Part 2 (Forward) of DCMD_SkateFormer added!")

Appending to /kaggle/working/models/dcmd_skateformer.py


# DCMD + SkateFormer Integration (Main Model) - Part 3 (Training)

In [13]:
%%writefile -a /kaggle/working/models/dcmd_skateformer.py

    def training_step(self, batch: List[torch.Tensor], batch_idx: int) -> torch.float32:
        """
        THAY ĐỔI QUAN TRỌNG:
        - Không dùng minimax strategy (series_loss, prior_loss)
        - Training đơn giản hơn, nhanh hơn
        """
        
        tensor_data, _ = self._unpack_data(batch)
        
        history_data = tensor_data[:, :, :self.n_his, :]
        x_0 = tensor_data
        
        # 1. Reconstruction loss
        condition_embedding, rec_his_data = self.rec_model(history_data)
        rec_loss = torch.mean(self.loss_fn(rec_his_data, history_data))
        self.log('rec_loss', rec_loss, prog_bar=True)
        
        # 2. Prediction loss
        dct_m, _ = get_dct_matrix(self.n_frames)
        dct_m_all = dct_m.float().to(self.device)
        x = x_0.permute(0, 2, 3, 1).contiguous()
        x = x.reshape([x.shape[0], self.n_frames, -1])
        y_0 = torch.matmul(dct_m_all, x)
        
        # Sample timesteps and add noise
        t = self.noise_scheduler.sample_timesteps(y_0.shape[0]).to(self.device)
        y_t, pre_noise = self.noise_scheduler.noise_motion(y_0, t)
        
        # Predict noise với SkateFormer
        pre_predicted_noise, _, _, _ = self.pre_model(
            y_t, t, condition_data=condition_embedding
        )
        
        pre_loss = torch.mean(self.loss_fn(pre_predicted_noise, pre_noise))
        self.log('pre_loss', pre_loss, prog_bar=True)
        
        # Total loss (simplified)
        total_loss = rec_loss + pre_loss
        self.log('train_loss', total_loss, prog_bar=True)
        
        return total_loss
    
    def configure_optimizers(self) -> Dict:
        """
        THAY ĐỔI QUAN TRỌNG:
        - Dùng AdamW với weight decay
        - Cosine annealing schedule
        """
        optimizer = torch.optim.AdamW(
            self.parameters(), 
            lr=self.learning_rate, 
            weight_decay=1e-4
        )
        
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, 
            T_max=self.trainer.max_epochs if hasattr(self, 'trainer') else 15,
            eta_min=1e-6
        )
        
        return {
            'optimizer': optimizer, 
            'lr_scheduler': {
                'scheduler': scheduler,
                'interval': 'epoch',
            }
        }
    
    def test_step(self, batch: List[torch.Tensor], batch_idx: int) -> None:
        self._test_output_list.append(self.forward(batch))
    
    def on_test_epoch_start(self) -> None:
        super().on_test_epoch_start()
        self._test_output_list = []
    
    def on_test_epoch_end(self) -> float:
        out, gt_data, trans, meta, frames = processing_data(self._test_output_list)
        del self._test_output_list
        
        if self.save_tensors:
            tensors = {
                'prediction': out, 'gt_data': gt_data,
                'trans': trans, 'metadata': meta, 'frames': frames
            }
            self._save_tensors(tensors, split_name=self.split,
                             aggr_strategy=self.aggregation_strategy,
                             n_gen=self.n_generated_samples)
        
        auc_score = self.post_processing(out, gt_data, trans, meta, frames)
        self.log('AUC', auc_score)
        print(f"\n🎯 Test AUC: {auc_score:.4f}")
        return auc_score

print("✅ Part 3 (Training) of DCMD_SkateFormer added!")

Appending to /kaggle/working/models/dcmd_skateformer.py


# DCMD + SkateFormer Integration (Main Model) - Part 4 (Helper Methods)

In [14]:
%%writefile -a /kaggle/working/models/dcmd_skateformer.py

    def _set_diffusion_variables(self) -> None:
        self.noise_scheduler = Diffusion(
            noise_steps=self.noise_steps,
            n_joints=self.n_joints,
            device=self.device,
            time=self.n_frames
        )
        self._beta_ = self.noise_scheduler.schedule_noise()
        self._alpha_ = (1. - self._beta_)
        self._alpha_hat_ = torch.cumprod(self._alpha_, dim=0)
    
    def _unpack_data(self, x: torch.Tensor) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        tensor_data = x[0].to(self.device)
        transformation_idx = x[1]
        metadata = x[2]
        actual_frames = x[3]
        meta_out = [transformation_idx, metadata, actual_frames]
        return tensor_data, meta_out
    
    def _aggregation_strategy(self, generated_xs: List[torch.Tensor], 
                            input_sequence: torch.Tensor, 
                            aggr_strategy: str) -> Tuple[torch.Tensor]:
        aggr_strategy = self.aggregation_strategy if aggr_strategy is None else aggr_strategy
        
        if aggr_strategy == 'random':
            return generated_xs[np.random.randint(len(generated_xs))], None
        
        B, repr_shape = input_sequence.shape[0], input_sequence.shape[1:]
        compute_loss = lambda x: torch.mean(
            self.loss_fn(x, input_sequence).reshape(-1, prod(repr_shape)), dim=-1
        )
        losses = [compute_loss(x) for x in generated_xs]
        
        if aggr_strategy == 'best':
            loss_of_selected_x = torch.full((B,), fill_value=1e10, device=self.device)
            selected_x = torch.zeros((B, *repr_shape)).to(self.device)
            
            for i in range(len(generated_xs)):
                mask = losses[i] < loss_of_selected_x
                loss_of_selected_x[mask] = losses[i][mask]
                selected_x[mask] = generated_xs[i][mask]
        elif aggr_strategy == 'mean':
            selected_x = None
            loss_of_selected_x = torch.mean(torch.stack(losses), dim=0)
        else:
            loss_of_selected_x = torch.mean(torch.stack(losses), dim=0)
            selected_x = None
        
        return selected_x, loss_of_selected_x
    
    def _pack_out_data(self, selected_x: torch.Tensor, loss_of_selected_x: torch.Tensor,
                      additional_out: List[torch.Tensor], return_: str) -> List[torch.Tensor]:
        if return_ is None:
            return_ = self.model_return_value
        
        out = []
        if return_ == 'loss' or return_ == 'all':
            if loss_of_selected_x is not None:
                out.append(loss_of_selected_x)
        if return_ == 'poses' or return_ == 'all':
            if selected_x is not None:
                out.append(selected_x)
        
        return out + additional_out
    
    def _save_tensors(self, tensors: Dict[str, torch.Tensor], split_name: str,
                     aggr_strategy: str, n_gen: int) -> None:
        name = f'saved_tensors_{split_name}_{aggr_strategy}_{n_gen}'
        path = os.path.join(self.ckpt_dir, name)
        os.makedirs(path, exist_ok=True)
        for t_name, tensor in tensors.items():
            torch.save(tensor, os.path.join(path, t_name + '.pt'))
    
    @property
    def _beta(self) -> torch.Tensor:
        return self._beta_.to(self.device)
    
    @property
    def _alpha(self) -> torch.Tensor:
        return self._alpha_.to(self.device)
    
    @property
    def _alpha_hat(self) -> torch.Tensor:
        return self._alpha_hat_.to(self.device)

print("✅ Part 4 (Helper Methods) of DCMD_SkateFormer added!")

Appending to /kaggle/working/models/dcmd_skateformer.py


# DCMD + SkateFormer Integration (Main Model) - Part 5 (Post-processing)

In [15]:
%%writefile -a /kaggle/working/models/dcmd_skateformer.py

    def post_processing(self, out: np.ndarray, gt_data: np.ndarray, 
                       trans: np.ndarray, meta: np.ndarray, 
                       frames: np.ndarray) -> float:
        """Post processing for AUC calculation"""
        
        all_gts = [f for f in os.listdir(self.gt_path) if f.endswith('.npy')]
        all_gts = sorted(all_gts)
        scene_clips = [(int(fn.split('_')[0]), int(fn.split('_')[1].split('.')[0])) 
                       for fn in all_gts]
        
        hr_ubnormal_masked_clips = get_hr_ubnormal_mask(self.split) \
            if (self.use_hr and (self.dataset_name == 'UBnormal')) else {}
        hr_avenue_masked_clips = get_avenue_mask() \
            if self.dataset_name == 'HR-Avenue' else {}
        
        num_transform = self.num_transforms
        model_scores_transf = {}
        dataset_gt_transf = {}
        
        print("\n🔄 Processing transformations...")
        for transformation in tqdm(range(num_transform)):
            dataset_gt = []
            model_scores = []
            cond_transform = (trans == transformation)
            
            out_transform, gt_data_transform, meta_transform, frames_transform = \
                filter_vectors_by_cond([out, gt_data, meta, frames], cond_transform)
            
            for idx in range(len(all_gts)):
                scene_idx, clip_idx = scene_clips[idx]
                gt = np.load(os.path.join(self.gt_path, all_gts[idx]))
                n_frames = gt.shape[0]
                
                cond_scene_clip = (meta_transform[:, 0] == scene_idx) & \
                                 (meta_transform[:, 1] == clip_idx)
                out_scene_clip, gt_scene_clip, meta_scene_clip, frames_scene_clip = \
                    filter_vectors_by_cond([out_transform, gt_data_transform,
                                           meta_transform, frames_transform],
                                          cond_scene_clip)
                
                figs_ids = sorted(list(set(meta_scene_clip[:, 2])))
                error_per_person = []
                
                for fig in figs_ids:
                    cond_fig = (meta_scene_clip[:, 2] == fig)
                    out_fig, _, frames_fig = filter_vectors_by_cond(
                        [out_scene_clip, gt_scene_clip, frames_scene_clip], cond_fig
                    )
                    loss_matrix = compute_var_matrix(out_fig, frames_fig, n_frames)
                    fig_reconstruction_loss = np.nanmax(loss_matrix, axis=0)
                    
                    if self.anomaly_score_pad_size != -1:
                        fig_reconstruction_loss = pad_scores(
                            fig_reconstruction_loss, gt, self.anomaly_score_pad_size
                        )
                    
                    error_per_person.append(fig_reconstruction_loss)
                
                clip_score = np.stack(error_per_person, axis=0)
                clip_score_log = np.log1p(clip_score)
                clip_score = np.mean(clip_score, axis=0) + \
                            (np.amax(clip_score_log, axis=0) - np.amin(clip_score_log, axis=0))
                
                # Mask for HR datasets
                if (scene_idx, clip_idx) in hr_ubnormal_masked_clips:
                    clip_score = clip_score[hr_ubnormal_masked_clips[(scene_idx, clip_idx)]]
                    gt = gt[hr_ubnormal_masked_clips[(scene_idx, clip_idx)]]
                
                if clip_idx in hr_avenue_masked_clips:
                    clip_score = clip_score[np.array(hr_avenue_masked_clips[clip_idx])==1]
                    gt = gt[np.array(hr_avenue_masked_clips[clip_idx])==1]
                
                # Score processing
                clip_score = score_process(
                    clip_score, self.anomaly_score_frames_shift, 
                    self.anomaly_score_filter_kernel_size
                )
                model_scores.append(clip_score)
                dataset_gt.append(gt)
            
            model_scores = np.concatenate(model_scores, axis=0)
            dataset_gt = np.concatenate(dataset_gt, axis=0)
            
            model_scores_transf[transformation] = model_scores
            dataset_gt_transf[transformation] = dataset_gt
        
        # Aggregate scores across transformations
        pds = np.mean(np.stack(list(model_scores_transf.values()), 0), 0)
        gt = dataset_gt_transf[0]
        
        # Calculate AUC
        auc = roc_auc_score(gt, pds)
        
        print(f"\n✨ Final AUC: {auc:.4f}")
        return auc

print("✅ Part 5 (Post-processing) - DCMD_SkateFormer COMPLETED!")

Appending to /kaggle/working/models/dcmd_skateformer.py


# Training Config 

In [16]:
%%writefile /kaggle/working/config/Avenue/skateformer_train.yaml
### SkateFormer + DCMD Training Configuration

## General settings
split: 'train'
debug: false
seed: 999
validation: false
use_hr: false

## Computational resources
accelerator: 'gpu'
devices: [0]

## Paths
dir_name: 'skateformer_train'
data_dir: '/kaggle/input/avenue/Avenue'
exp_dir: '/kaggle/working/checkpoints/HR-Avenue'
test_path: '/kaggle/input/avenue/Avenue/testing/test_frame_mask'
gt_path: '/kaggle/input/avenue/Avenue/testing/test_frame_mask'
load_ckpt: 'best.ckpt'
create_experiment_dir: true

## WANDB configuration
use_wandb: false
project_name: "dcmd_skateformer"
wandb_entity: "ntrduc1310"
group_name: "avenue_experiments"
use_ema: false

### Model configuration (SkateFormer)
dropout: 0.1
conditioning_strategy: 'inject'
h_dim: 256
latent_dim: 256
channels: [256, 128, 256]

## Diffusion configuration
noise_steps: 10

### Training configuration (OPTIMIZED)
n_epochs: 12
opt_lr: 0.0003

## Loss configuration
loss_fn: 'smooth_l1'

### Inference configuration
n_generated_samples: 30
model_return_value: 'loss'
aggregation_strategy: 'best'
filter_kernel_size: 30
frames_shift: 6
save_tensors: true
load_tensors: false

### Dataset configuration
dataset_choice: 'HR-Avenue'
seg_len: 7
vid_res: [640, 360]
batch_size: 4096
pad_size: 12

headless: false
hip_center: false
kp18_format: false
normalization_strategy: 'robust'
num_coords: 2
num_transform: 5
num_workers: 4
seg_stride: 1
seg_th: 0
start_offset: 0
symm_range: true
use_fitted_scaler: false

n_his: 3
padding: 'LastFrame'

## SkateFormer specific
num_layers: 4
num_heads: 8
latent_dims: 512

Writing /kaggle/working/config/Avenue/skateformer_train.yaml


# Test Config

In [17]:
%%writefile /kaggle/working/config/Avenue/skateformer_test.yaml
### SkateFormer + DCMD Test Configuration

split: 'test'
debug: false
seed: 999
validation: false
use_hr: false

accelerator: 'gpu'
devices: [0]

dir_name: 'skateformer_test'
data_dir: '/kaggle/input/avenue/Avenue'
exp_dir: '/kaggle/working/checkpoints/HR-Avenue'
test_path: '/kaggle/input/avenue/Avenue/testing/test_frame_mask'
gt_path: '/kaggle/input/avenue/Avenue/testing/test_frame_mask'
load_ckpt: 'last.ckpt'
create_experiment_dir: false

use_wandb: false
use_ema: false

dropout: 0.1
conditioning_strategy: 'inject'
h_dim: 256
latent_dim: 256
channels: [256, 128, 256]

noise_steps: 10
n_epochs: 12
opt_lr: 0.0003
loss_fn: 'smooth_l1'

n_generated_samples: 30
model_return_value: 'loss'
aggregation_strategy: 'best'
filter_kernel_size: 30
frames_shift: 6
save_tensors: true
load_tensors: false

dataset_choice: 'HR-Avenue'
seg_len: 7
vid_res: [640, 360]
batch_size: 4096
pad_size: 12

headless: false
hip_center: false
kp18_format: false
normalization_strategy: 'robust'
num_coords: 2
num_transform: 5
num_workers: 4
seg_stride: 1
seg_th: 0
start_offset: 0
symm_range: true
use_fitted_scaler: false

n_his: 3
padding: 'LastFrame'

num_layers: 4
num_heads: 8
latent_dims: 512

Writing /kaggle/working/config/Avenue/skateformer_test.yaml


# Training Script

In [18]:
%%writefile /kaggle/working/train_skateformer.py
import argparse
import os
import random
import numpy as np
import pytorch_lightning as pl
import torch
import yaml

import sys
sys.path.append('/kaggle/working/models')
sys.path.append('/kaggle/working')

from dcmd_skateformer import DCMD_SkateFormer
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.strategies import DDPStrategy
from utils.argparser import init_args
from utils.dataset import get_dataset_and_loader

if __name__ == '__main__':
    print("🚀 Starting SkateFormer + DCMD Training...")
    
    parser = argparse.ArgumentParser(description='SkateFormer_DCMD')
    parser.add_argument('-c', '--config', type=str, required=True)
    
    args = parser.parse_args()
    config_path = args.config
    args = yaml.load(open(args.config), Loader=yaml.FullLoader)
    args = argparse.Namespace(**args)
    args = init_args(args)
    
    # Copy config
    os.system(f'cp {config_path} {os.path.join(args.ckpt_dir, "config.yaml")}')
    
    # Set seeds
    torch.manual_seed(args.seed)
    random.seed(args.seed)
    np.random.seed(args.seed)
    pl.seed_everything(args.seed)
    
    print(f"✅ Seed set to {args.seed}")
    
    # Callbacks
    callbacks = [
        ModelCheckpoint(
            dirpath=args.ckpt_dir,
            filename='best-{epoch:02d}-{train_loss:.4f}',
            save_top_k=2,
            save_last=True,
            monitor='train_loss',
            mode='min'
        ),
        EarlyStopping(
            monitor='train_loss',
            patience=5,
            mode='min'
        )
    ]
    
    print("✅ Callbacks configured")
    
    # Data
    print("📊 Loading data...")
    _, train_loader, _, val_loader = get_dataset_and_loader(
        args, split=args.split, validation=args.validation
    )
    print(f"✅ Data loaded: {len(train_loader)} batches")
    
    # Model
    print("🔧 Building model...")
    model = DCMD_SkateFormer(args)
    
    # Trainer
    trainer = pl.Trainer(
        accelerator=args.accelerator,
        devices=args.devices,
        default_root_dir=args.ckpt_dir,
        max_epochs=args.n_epochs,
        logger=False,
        callbacks=callbacks,
        log_every_n_steps=20,
        num_sanity_val_steps=0,
        deterministic=True,
        gradient_clip_val=1.0,
        precision='16-mixed'  # Mixed precision for speed
    )
    
    print("🏋️ Starting training...")
    trainer.fit(model=model, train_dataloaders=train_loader)
    
    print("✅ Training completed!")

Writing /kaggle/working/train_skateformer.py


# Test Script

In [19]:
%%writefile /kaggle/working/test_skateformer.py
import argparse
import os
import pytorch_lightning as pl
import yaml

import sys
sys.path.append('/kaggle/working/models')
sys.path.append('/kaggle/working')

from dcmd_skateformer import DCMD_SkateFormer
from utils.argparser import init_args
from utils.dataset import get_dataset_and_loader

if __name__ == '__main__':
    print("🧪 Starting SkateFormer + DCMD Testing...")
    
    parser = argparse.ArgumentParser(description='Test SkateFormer_DCMD')
    parser.add_argument('-c', '--config', type=str, required=True)
    
    args = parser.parse_args()
    args = yaml.load(open(args.config), Loader=yaml.FullLoader)
    args = argparse.Namespace(**args)
    args = init_args(args)
    
    model = DCMD_SkateFormer(args)
    
    if args.load_tensors:
        print("📂 Loading saved tensors...")
        model.test_on_saved_tensors(split_name=args.split)
    else:
        print("📊 Loading test data...")
        
        # IMPORTANT: Update this path to your checkpoint
        ckpt_path = os.path.join(args.exp_dir, 'skateformer_train', 'last.ckpt')
        
        if not os.path.exists(ckpt_path):
            print(f"❌ Checkpoint not found: {ckpt_path}")
            print("Available files:")
            os.system(f"ls -la {os.path.join(args.exp_dir, 'skateformer_train')}")
        else:
            dataset, loader, _, _ = get_dataset_and_loader(args, split=args.split)
            
            trainer = pl.Trainer(
                accelerator=args.accelerator,
                devices=args.devices[:1],
                default_root_dir=args.ckpt_dir,
                max_epochs=1,
                logger=False
            )
            
            print("🔬 Running test...")
            out = trainer.test(model, dataloaders=loader, ckpt_path=ckpt_path)
            print(f"\n🎯 Final Test AUC: {out[0]['AUC']:.4f}")

Writing /kaggle/working/test_skateformer.py


**Sửa lỗi IndentationError trong dcmd_skateformer.py**


In [20]:
%%writefile /kaggle/working/models/dcmd_skateformer.py
import argparse
import os
from math import prod
from typing import Dict, List, Tuple
import numpy as np
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

# Import SkateFormer components
import sys
sys.path.append('/kaggle/working/models')
from skateformer_encoder import SkateFormerEncoder
from skateformer_diffusion import SkateFormerDiffusion

# Import DCMD utilities
sys.path.append('/kaggle/working')
from utils.diffusion_utils import Diffusion
from utils.eval_utils import (compute_var_matrix, filter_vectors_by_cond,
                              get_avenue_mask, get_hr_ubnormal_mask, pad_scores, score_process)
from utils.model_utils import processing_data
from utils.tools import get_dct_matrix, generate_pad, padding_traj


class DCMD_SkateFormer(pl.LightningModule):
    """
    DCMD model với SkateFormer architecture
    """
    
    losses = {'l1': nn.L1Loss, 'smooth_l1': nn.SmoothL1Loss, 'mse': nn.MSELoss}
    
    def __init__(self, args: argparse.Namespace) -> None:
        super(DCMD_SkateFormer, self).__init__()
        
        self.save_hyperparameters(args)
        
        # Data parameters
        self.n_frames = args.seg_len
        self.num_coords = args.num_coords
        self.n_joints = 17  # Avenue dataset
        self.n_his = args.n_his
        
        # Training parameters
        self.learning_rate = args.opt_lr
        self.loss_fn = self.losses[args.loss_fn](reduction='none')
        self.noise_steps = args.noise_steps
        self.aggregation_strategy = args.aggregation_strategy
        self.n_generated_samples = args.n_generated_samples
        self.model_return_value = args.model_return_value
        
        # Other parameters
        self.gt_path = args.gt_path
        self.split = args.split
        self.ckpt_dir = args.ckpt_dir
        self.save_tensors = args.save_tensors
        self.num_transforms = args.num_transform
        self.anomaly_score_pad_size = args.pad_size
        self.anomaly_score_filter_kernel_size = args.filter_kernel_size
        self.anomaly_score_frames_shift = args.frames_shift
        self.dataset_name = args.dataset_choice
        self.padding = args.padding
        
        # Generate padding indices
        self.idx_pad, self.zero_index = generate_pad(
            self.padding, self.n_his, self.n_frames - self.n_his
        )
        
        # Diffusion setup
        self._set_diffusion_variables()
        
        # Build models
        self.build_model()
    
    def build_model(self) -> None:
        """Build SkateFormer-based encoder and diffusion model"""
        
        print("🔨 Building SkateFormer models...")
        
        # SkateFormer Encoder (thay thế STSAE)
        self.rec_model = SkateFormerEncoder(
            num_coords=self.num_coords,
            num_joints=self.n_joints,
            n_frames=self.n_his,
            embed_dim=256,
            depth=3,  # Reduced depth
            num_heads=8,
            mlp_ratio=4.,
            drop_rate=0.1,
            attn_drop_rate=0.1
        )
        
        # SkateFormer Diffusion Model (thay thế MotionTransformer)
        self.pre_model = SkateFormerDiffusion(
            input_feats=2 * self.n_joints,
            num_frames=self.n_frames,
            latent_dim=512,
            depth=4,  # Reduced depth
            num_heads=8,
            num_joints=self.n_joints,
            dropout=0.1,
            inject_condition=True
        )
        
        print("✅ Models built successfully!")
    
    def forward(self, input_data: List[torch.Tensor], aggr_strategy: str = None, 
                return_: str = None) -> List[torch.Tensor]:
        """Forward pass"""
        
        tensor_data, meta_out = self._unpack_data(input_data)
        B = tensor_data.shape[0]
        
        # Split history and full sequence
        history_data = tensor_data[:, :, :self.n_his, :]
        x_0 = padding_traj(history_data, self.padding, self.idx_pad, self.zero_index)
        
        generated_xs = []
        
        for _ in range(self.n_generated_samples):
            # 1. Reconstruction với SkateFormer Encoder
            condition_embedding, rec_his_data = self.rec_model(history_data)
            
            # 2. DCT transformation
            dct_m, idct_m = get_dct_matrix(self.n_frames)
            dct_m_all = dct_m.float().to(self.device)
            idct_m_all = idct_m.float().to(self.device)
            
            x = x_0.permute(0, 2, 3, 1).contiguous()
            x = x.reshape([x.shape[0], self.n_frames, -1])
            y = torch.matmul(dct_m_all, x)
            
            # 3. Diffusion process
            y_d = torch.randn_like(y, device=self.device)
            
            for i in reversed(range(1, self.noise_steps)):
                t = torch.full(size=(B,), fill_value=i, dtype=torch.long, device=self.device)
                t_prev = torch.full(size=(B,), fill_value=i, dtype=torch.long, device=self.device)
                t_prev[0] = 0
                
                noise_pre = torch.randn_like(y_d) if i > 1 else torch.zeros_like(y_d)
                
                # SkateFormer prediction
                predicted_noise_pre, _, _, _ = self.pre_model(
                    y_d, t, condition_data=condition_embedding
                )
                
                alpha_pre = self._alpha[t][:, None, None]
                alpha_hat_pre = self._alpha_hat[t][:, None, None]
                beta_pre = self._beta[t][:, None, None]
                
                y_d = (1 / torch.sqrt(alpha_pre)) * \
                      (y_d - ((1 - alpha_pre) / (torch.sqrt(1 - alpha_hat_pre))) * predicted_noise_pre) + \
                      torch.sqrt(beta_pre) * noise_pre
                
                alpha_hat_prev = self._alpha_hat[t_prev][:, None, None]
                y_n = (torch.sqrt(alpha_hat_prev) * y) + (torch.sqrt(1 - alpha_hat_prev) * noise_pre)
                
                # Mask completion
                mask = torch.zeros_like(x, device=self.device)
                for m in range(0, self.n_his):
                    mask[:, m, :] = 1
                
                y_d_idct = torch.matmul(idct_m_all, y_d)
                y_n_idct = torch.matmul(idct_m_all, y_n)
                m_mul_y_n = torch.mul(mask, y_n_idct)
                m_mul_y_d = torch.mul((1 - mask), y_d_idct)
                m_y = m_mul_y_d + m_mul_y_n
                y_d = torch.matmul(dct_m_all, m_y)
            
            # iDCT
            pre_future_data = torch.matmul(idct_m_all, y_d)
            pre_future_data = pre_future_data.reshape(pre_future_data.shape[0], pre_future_data.shape[1], -1, 2)
            pre_future_data = pre_future_data.permute(0, 3, 1, 2).contiguous()
            pre_future_data = pre_future_data[:, :, self.n_his:, :]
            
            xs = torch.cat((rec_his_data, pre_future_data), dim=2)
            generated_xs.append(xs)
        
        selected_x, loss_of_selected_x = self._aggregation_strategy(
            generated_xs, tensor_data, aggr_strategy
        )
        
        return self._pack_out_data(selected_x, loss_of_selected_x, 
                                   [tensor_data] + meta_out, return_=return_)
    
    def training_step(self, batch: List[torch.Tensor], batch_idx: int) -> torch.float32:
        """Training step - FIXED INDENTATION"""
        
        opt = self.optimizers()
        tensor_data, _ = self._unpack_data(batch)
        
        history_data = tensor_data[:, :, :self.n_his, :]
        x_0 = tensor_data
        
        # Reconstruction
        condition_embedding, rec_his_data = self.rec_model(history_data)
        rec_loss = torch.mean(self.loss_fn(rec_his_data, history_data))
        self.log('rec_loss', rec_loss, prog_bar=True)
        
        # Prediction
        dct_m, _ = get_dct_matrix(self.n_frames)
        dct_m_all = dct_m.float().to(self.device)
        x = x_0.permute(0, 2, 3, 1).contiguous()
        x = x.reshape([x.shape[0], self.n_frames, -1])
        y_0 = torch.matmul(dct_m_all, x)
        
        t = self.noise_scheduler.sample_timesteps(y_0.shape[0]).to(self.device)
        y_t, pre_noise = self.noise_scheduler.noise_motion(y_0, t)
        
        pre_predicted_noise, _, _, _ = self.pre_model(
            y_t, t, condition_data=condition_embedding
        )
        
        pre_loss = torch.mean(self.loss_fn(pre_predicted_noise, pre_noise))
        self.log('pre_loss', pre_loss, prog_bar=True)
        
        # Total loss
        total_loss = rec_loss + pre_loss
        self.log('train_loss', total_loss, prog_bar=True)
        
        return total_loss
    
    def configure_optimizers(self) -> Dict:
        """Configure optimizer"""
        optimizer = Adam(self.parameters(), lr=self.learning_rate, weight_decay=1e-5)
        
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=self.trainer.max_epochs, eta_min=1e-6
        )
        
        return {
            'optimizer': optimizer, 
            'lr_scheduler': {
                'scheduler': scheduler,
                'interval': 'epoch',
                'monitor': 'train_loss'
            }
        }
    
    def test_step(self, batch: List[torch.Tensor], batch_idx: int) -> None:
        """Test step"""
        self._test_output_list.append(self.forward(batch))
    
    def on_test_epoch_start(self) -> None:
        """Initialize test output list"""
        super().on_test_epoch_start()
        self._test_output_list = []
    
    def on_test_epoch_end(self) -> float:
        """Process test outputs"""
        out, gt_data, trans, meta, frames = processing_data(self._test_output_list)
        del self._test_output_list
        
        if self.save_tensors:
            tensors = {
                'prediction': out, 'gt_data': gt_data,
                'trans': trans, 'metadata': meta, 'frames': frames
            }
            self._save_tensors(tensors, split_name=self.split,
                             aggr_strategy=self.aggregation_strategy,
                             n_gen=self.n_generated_samples)
        
        auc_score = self.post_processing(out, gt_data, trans, meta, frames)
        self.log('AUC', auc_score)
        return auc_score
    
    def post_processing(self, out: np.ndarray, gt_data: np.ndarray, trans: np.ndarray,
                       meta: np.ndarray, frames: np.ndarray) -> float:
        """Post processing để tính AUC"""
        
        all_gts = [file_name for file_name in os.listdir(self.gt_path) if file_name.endswith('.npy')]
        all_gts = sorted(all_gts)
        scene_clips = [(int(fn.split('_')[0]), int(fn.split('_')[1].split('.')[0])) for fn in all_gts]
        
        hr_ubnormal_masked_clips = get_hr_ubnormal_mask(self.split) if (self.use_hr and (self.dataset_name == 'UBnormal')) else {}
        hr_avenue_masked_clips = get_avenue_mask() if self.dataset_name == 'HR-Avenue' else {}
        
        num_transform = self.num_transforms
        model_scores_transf = {}
        dataset_gt_transf = {}
        
        for transformation in tqdm(range(num_transform), desc="Processing transformations"):
            dataset_gt = []
            model_scores = []
            cond_transform = (trans == transformation)
            
            out_transform, gt_data_transform, meta_transform, frames_transform = filter_vectors_by_cond(
                [out, gt_data, meta, frames], cond_transform
            )
            
            for idx in range(len(all_gts)):
                scene_idx, clip_idx = scene_clips[idx]
                
                gt = np.load(os.path.join(self.gt_path, all_gts[idx]))
                n_frames = gt.shape[0]
                
                cond_scene_clip = (meta_transform[:, 0] == scene_idx) & (meta_transform[:, 1] == clip_idx)
                out_scene_clip, gt_scene_clip, meta_scene_clip, frames_scene_clip = filter_vectors_by_cond(
                    [out_transform, gt_data_transform, meta_transform, frames_transform],
                    cond_scene_clip
                )
                
                figs_ids = sorted(list(set(meta_scene_clip[:, 2])))
                error_per_person = []
                
                for fig in figs_ids:
                    cond_fig = (meta_scene_clip[:, 2] == fig)
                    out_fig, _, frames_fig = filter_vectors_by_cond(
                        [out_scene_clip, gt_scene_clip, frames_scene_clip], cond_fig
                    )
                    loss_matrix = compute_var_matrix(out_fig, frames_fig, n_frames)
                    fig_reconstruction_loss = np.nanmax(loss_matrix, axis=0)
                    
                    if self.anomaly_score_pad_size != -1:
                        fig_reconstruction_loss = pad_scores(fig_reconstruction_loss, gt, self.anomaly_score_pad_size)
                    
                    error_per_person.append(fig_reconstruction_loss)
                
                clip_score = np.stack(error_per_person, axis=0)
                clip_score_log = np.log1p(clip_score)
                clip_score = np.mean(clip_score, axis=0) + (np.amax(clip_score_log, axis=0) - np.amin(clip_score_log, axis=0))
                
                if (scene_idx, clip_idx) in hr_ubnormal_masked_clips:
                    clip_score = clip_score[hr_ubnormal_masked_clips[(scene_idx, clip_idx)]]
                    gt = gt[hr_ubnormal_masked_clips[(scene_idx, clip_idx)]]
                
                if clip_idx in hr_avenue_masked_clips:
                    clip_score = clip_score[np.array(hr_avenue_masked_clips[clip_idx]) == 1]
                    gt = gt[np.array(hr_avenue_masked_clips[clip_idx]) == 1]
                
                clip_score = score_process(clip_score, self.anomaly_score_frames_shift, 
                                         self.anomaly_score_filter_kernel_size)
                model_scores.append(clip_score)
                dataset_gt.append(gt)
            
            model_scores = np.concatenate(model_scores, axis=0)
            dataset_gt = np.concatenate(dataset_gt, axis=0)
            
            model_scores_transf[transformation] = model_scores
            dataset_gt_transf[transformation] = dataset_gt
        
        # Aggregate scores
        pds = np.mean(np.stack(list(model_scores_transf.values()), 0), 0)
        gt = dataset_gt_transf[0]
        
        # Compute AUC
        auc = roc_auc_score(gt, pds)
        
        return auc
    
    # Helper methods
    
    def _set_diffusion_variables(self) -> None:
        """Set diffusion variables"""
        self.noise_scheduler = Diffusion(
            noise_steps=self.noise_steps,
            n_joints=self.n_joints,
            device=self.device,
            time=self.n_frames
        )
        self._beta_ = self.noise_scheduler.schedule_noise()
        self._alpha_ = (1. - self._beta_)
        self._alpha_hat_ = torch.cumprod(self._alpha_, dim=0)
    
    def _unpack_data(self, x: torch.Tensor) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        """Unpack data"""
        tensor_data = x[0].to(self.device)
        transformation_idx = x[1]
        metadata = x[2]
        actual_frames = x[3]
        meta_out = [transformation_idx, metadata, actual_frames]
        return tensor_data, meta_out
    
    def _aggregation_strategy(self, generated_xs: List[torch.Tensor], 
                            input_sequence: torch.Tensor, 
                            aggr_strategy: str) -> Tuple[torch.Tensor]:
        """Aggregate generated samples"""
        aggr_strategy = self.aggregation_strategy if aggr_strategy is None else aggr_strategy
        
        if aggr_strategy == 'random':
            idx = np.random.randint(len(generated_xs))
            return generated_xs[idx], None
        
        B, repr_shape = input_sequence.shape[0], input_sequence.shape[1:]
        compute_loss = lambda x: torch.mean(
            self.loss_fn(x, input_sequence).reshape(-1, prod(repr_shape)), dim=-1
        )
        losses = [compute_loss(x) for x in generated_xs]
        
        if aggr_strategy == 'best':
            loss_of_selected_x = torch.full((B,), fill_value=1e10, device=self.device)
            selected_x = torch.zeros((B, *repr_shape)).to(self.device)
            
            for i in range(len(generated_xs)):
                mask = losses[i] < loss_of_selected_x
                loss_of_selected_x[mask] = losses[i][mask]
                selected_x[mask] = generated_xs[i][mask]
        elif aggr_strategy == 'mean':
            selected_x = None
            loss_of_selected_x = torch.mean(torch.stack(losses), dim=0)
        else:
            loss_of_selected_x = torch.mean(torch.stack(losses), dim=0)
            selected_x = None
        
        return selected_x, loss_of_selected_x
    
    def _pack_out_data(self, selected_x: torch.Tensor, loss_of_selected_x: torch.Tensor,
                      additional_out: List[torch.Tensor], return_: str) -> List[torch.Tensor]:
        """Pack output data"""
        if return_ is None:
            return_ = self.model_return_value
        
        out = []
        if return_ == 'poses' and selected_x is not None:
            out = [selected_x]
        elif return_ == 'loss' and loss_of_selected_x is not None:
            out = [loss_of_selected_x]
        elif return_ == 'all':
            if loss_of_selected_x is not None:
                out.append(loss_of_selected_x)
            if selected_x is not None:
                out.append(selected_x)
        
        return out + additional_out
    
    def _save_tensors(self, tensors: Dict[str, torch.Tensor], split_name: str,
                     aggr_strategy: str, n_gen: int) -> None:
        """Save tensors"""
        name = f'saved_tensors_{split_name}_{aggr_strategy}_{n_gen}'
        path = os.path.join(self.ckpt_dir, name)
        os.makedirs(path, exist_ok=True)
        for t_name, tensor in tensors.items():
            torch.save(tensor, os.path.join(path, t_name + '.pt'))
    
    @property
    def _beta(self) -> torch.Tensor:
        return self._beta_.to(self.device)
    
    @property
    def _alpha(self) -> torch.Tensor:
        return self._alpha_.to(self.device)
    
    @property
    def _alpha_hat(self) -> torch.Tensor:
        return self._alpha_hat_.to(self.device)
    
    @property
    def use_hr(self):
        return getattr(self.hparams, 'use_hr', False)

print("✅ DCMD_SkateFormer model FIXED and created!")

Overwriting /kaggle/working/models/dcmd_skateformer.py


**Giảm Memory Usage - Fix SkateFormer Block**

In [21]:
%%writefile /kaggle/working/models/skateformer_block.py
import torch
import torch.nn as nn
import sys
sys.path.append('/kaggle/working/models')
from skateformer_attention import SpatialAttention, TemporalAttention


class Mlp(nn.Module):
    """MLP module - OPTIMIZED"""
    def __init__(self, in_features, hidden_features=None, out_features=None, 
                 act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class SkateFormerBlock(nn.Module):
    """SkateFormer Block - MEMORY OPTIMIZED"""
    def __init__(self, dim, num_heads, num_joints=17, mlp_ratio=2.,  # ⚠️ Giảm từ 4 xuống 2
                 qkv_bias=False, drop=0., attn_drop=0., 
                 attention_type='spatial'):
        super().__init__()
        self.attention_type = attention_type
        self.norm1 = nn.LayerNorm(dim)
        
        if attention_type == 'spatial':
            self.attn = SpatialAttention(
                dim, num_heads=num_heads, num_joints=num_joints,
                qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop
            )
        elif attention_type == 'temporal':
            self.attn = TemporalAttention(
                dim, num_heads=num_heads, num_joints=num_joints,
                qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop
            )
        
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, 
                       act_layer=nn.GELU, drop=drop)

    def forward(self, x):
        """x: [B, C, T, V] - OPTIMIZED để giảm memory"""
        B, C, T, V = x.shape
        
        # Attention with residual
        identity = x
        
        # ⚠️ In-place operations để tiết kiệm memory
        x = x.permute(0, 2, 3, 1).contiguous()
        x = self.norm1(x)
        x = x.permute(0, 3, 1, 2).contiguous()
        x = self.attn(x)
        x = x + identity  # Residual
        
        # MLP with residual
        identity = x
        x = x.permute(0, 2, 3, 1).contiguous()
        x = self.norm2(x)
        x = self.mlp(x)
        x = x.permute(0, 3, 1, 2).contiguous()
        x = x + identity  # Residual
        
        return x


class STBlock(nn.Module):
    """Spatial-Temporal Block - OPTIMIZED"""
    def __init__(self, dim, num_heads, num_joints=17, mlp_ratio=2.,  # ⚠️ Giảm ratio
                 qkv_bias=False, drop=0., attn_drop=0.):
        super().__init__()
        
        self.spatial_block = SkateFormerBlock(
            dim, num_heads, num_joints, mlp_ratio, 
            qkv_bias, drop, attn_drop, attention_type='spatial'
        )
        
        self.temporal_block = SkateFormerBlock(
            dim, num_heads, num_joints, mlp_ratio, 
            qkv_bias, drop, attn_drop, attention_type='temporal'
        )
    
    def forward(self, x):
        # ⚠️ Sequential để giảm peak memory
        x = self.spatial_block(x)
        x = self.temporal_block(x)
        return x

print("✅ OPTIMIZED SkateFormer Block created!")

Overwriting /kaggle/working/models/skateformer_block.py


**Giảm Memory - Fix SkateFormer Encoder**

In [22]:
%%writefile /kaggle/working/models/skateformer_encoder.py
import torch
import torch.nn as nn
import math
import sys
sys.path.append('/kaggle/working/models')
from skateformer_block import STBlock


class SkateFormerEncoder(nn.Module):
    """SkateFormer Encoder - MEMORY OPTIMIZED"""
    def __init__(self, num_coords=2, num_joints=17, n_frames=3, 
                 embed_dim=128, depth=2, num_heads=4, mlp_ratio=2.,  # ⚠️ GIẢM MẠNH
                 drop_rate=0.1, attn_drop_rate=0.1):
        super().__init__()
        self.num_coords = num_coords
        self.num_joints = num_joints
        self.n_frames = n_frames
        self.embed_dim = embed_dim
        
        # Input embedding
        self.input_embed = nn.Conv2d(num_coords, embed_dim, 
                                     kernel_size=1, stride=1)
        
        # Positional embedding
        self.pos_embed_temporal = nn.Parameter(
            torch.zeros(1, embed_dim, n_frames, 1)
        )
        self.pos_embed_spatial = nn.Parameter(
            torch.zeros(1, embed_dim, 1, num_joints)
        )
        
        # ⚠️ Giảm depth từ 3 xuống 2 blocks
        self.blocks = nn.ModuleList([
            STBlock(
                dim=embed_dim, num_heads=num_heads, num_joints=num_joints,
                mlp_ratio=mlp_ratio, qkv_bias=True, drop=drop_rate,
                attn_drop=attn_drop_rate
            )
            for _ in range(depth)
        ])
        
        # Global pooling and latent projection
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.latent_proj = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(256, embed_dim),
            nn.ReLU(),
        )
        self.output_proj = nn.Conv2d(embed_dim, num_coords, 
                                     kernel_size=1, stride=1)
        
        # Initialize
        nn.init.trunc_normal_(self.pos_embed_temporal, std=0.02)
        nn.init.trunc_normal_(self.pos_embed_spatial, std=0.02)

    def forward(self, x):
        B, C, T, V = x.shape
        
        # Input embedding
        x = self.input_embed(x)
        x = x + self.pos_embed_temporal + self.pos_embed_spatial
        
        # Apply ST blocks
        for block in self.blocks:
            x = block(x)
        
        # Extract latent
        latent_feat = self.global_pool(x).squeeze(-1).squeeze(-1)
        latent = self.latent_proj(latent_feat)
        
        # Decode
        dec_feat = self.decoder(latent)
        dec_feat = dec_feat.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, T, V)
        dec_feat = dec_feat + x  # Skip connection
        recon = self.output_proj(dec_feat)
        
        return latent, recon

print("✅ OPTIMIZED SkateFormer Encoder created!")

Overwriting /kaggle/working/models/skateformer_encoder.py


**Giảm Memory - Fix Config File**

In [23]:
%%writefile /kaggle/working/config/Avenue/skateformer_train.yaml
### MEMORY OPTIMIZED Configuration

split: 'train'
debug: false
seed: 999
validation: false
use_hr: false

accelerator: 'gpu'
devices: [0]

dir_name: 'skateformer_train'
data_dir: '/kaggle/input/avenue/Avenue'
exp_dir: '/kaggle/working/checkpoints/HR-Avenue'
test_path: '/kaggle/input/avenue/Avenue/testing/test_frame_mask'
load_ckpt: 'best.ckpt'
create_experiment_dir: true

use_wandb: false
use_ema: false

# ⚠️ MEMORY OPTIMIZATIONS
dropout: 0.1
conditioning_strategy: 'inject'
h_dim: 128          # Giảm từ 256
latent_dim: 256
channels: [128, 64, 128]  # Giảm kích thước

noise_steps: 10

# ⚠️ TRAINING OPTIMIZATIONS
n_epochs: 5          # Giảm epochs cho test
opt_lr: 0.001
loss_fn: 'smooth_l1'

# ⚠️ INFERENCE OPTIMIZATIONS
n_generated_samples: 10   # Giảm từ 30 xuống 10
model_return_value: 'loss'
aggregation_strategy: 'best'
filter_kernel_size: 30
frames_shift: 6
save_tensors: true
load_tensors: false

# ⚠️ DATASET OPTIMIZATIONS
dataset_choice: 'HR-Avenue'
seg_len: 7
vid_res: [640, 360]
batch_size: 256      # ⚠️ GIẢM TỪ 4096 XUỐNG 256 ĐỂ TIẾT KIỆM MEMORY
pad_size: 12

headless: false
hip_center: false
kp18_format: false
normalization_strategy: 'robust'
num_coords: 2
num_transform: 5
num_workers: 2       # ⚠️ Giảm workers
seg_stride: 1
seg_th: 0
start_offset: 0
symm_range: true
use_fitted_scaler: false

n_his: 3
padding: 'LastFrame'

# ⚠️ SKATEFORMER OPTIMIZATIONS
num_layers: 4        # Giảm từ 6
num_heads: 4         # Giảm từ 8
latent_dims: 256     # Giảm từ 512
partition_size: 4

Overwriting /kaggle/working/config/Avenue/skateformer_train.yaml


**Clear Memory & Restart**

In [24]:
import gc
import torch

# Clear CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
# Garbage collection
gc.collect()

# Check memory
if torch.cuda.is_available():
    print(f"🔋 GPU Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"🔋 GPU Memory Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"🔋 GPU Memory Cached: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")
    
print("✅ Memory cleared!")

🔋 GPU Memory Available: 15.83 GB
🔋 GPU Memory Allocated: 0.00 GB
🔋 GPU Memory Cached: 0.00 GB
✅ Memory cleared!


**Giảm Memory Usage**************

In [25]:
%%writefile /kaggle/working/config/Avenue/skateformer_train.yaml
### Experiment configuration với SkateFormer - OPTIMIZED FOR MEMORY

## General settings
split: 'train'
debug: false
seed: 999
validation: false
use_hr: false

## Computational resources
accelerator: 'gpu'
devices: [0]

## Paths
dir_name: 'skateformer_train'
data_dir: '/kaggle/input/avenue/Avenue'
exp_dir: '/kaggle/working/checkpoints'
test_path: '/kaggle/input/avenue/Avenue/testing/test_frame_mask'
load_ckpt: 'best.ckpt'
create_experiment_dir: true

## WANDB configuration
use_wandb: false
project_name: "dcmd_skateformer"
wandb_entity: "your_entity"
group_name: "avenue_experiments"
use_ema: false

### Model configuration - REDUCED SIZE

## SkateFormer parameters
dropout: 0.1
conditioning_strategy: 'inject'
h_dim: 128          # Giảm từ 256
latent_dim: 128     # Giảm từ 256
channels: [128, 64, 128]  # Giảm từ [256, 128, 256]
embed_dim: 128      # NEW - cho encoder

## Diffusion configuration
noise_steps: 8      # Giảm từ 10

### Training configuration
n_epochs: 8         # Giảm từ 15
opt_lr: 0.001
accumulate_grad_batches: 4  # Gradient accumulation

## Loss configuration
loss_fn: 'smooth_l1'

### Inference configuration
n_generated_samples: 20  # Giảm từ 30
model_return_value: 'loss'
aggregation_strategy: 'best'
filter_kernel_size: 30
frames_shift: 6
save_tensors: true
load_tensors: false

### Dataset configuration - OPTIMIZED
dataset_choice: 'HR-Avenue'
seg_len: 7
vid_res: [640, 360]
batch_size: 512     # GIẢM MẠNH từ 4096
pad_size: 12

headless: false
hip_center: false
kp18_format: false
normalization_strategy: 'robust'
num_coords: 2
num_transform: 5
num_workers: 2      # Giảm từ 8
seg_stride: 1
seg_th: 0
start_offset: 0
symm_range: true
use_fitted_scaler: false

n_his: 3
padding: 'LastFrame'

## SkateFormer specific - REDUCED
num_layers: 4       # Giảm từ 6
num_heads: 4        # Giảm từ 8
latent_dims: 256    # Giảm từ 512
encoder_depth: 2    # NEW - số ST blocks trong encoder
diffusion_depth: 3  # NEW - số ST blocks trong diffusion

Overwriting /kaggle/working/config/Avenue/skateformer_train.yaml


**Update SkateFormer Encoder (Memory Efficient)**

In [26]:
%%writefile /kaggle/working/models/skateformer_encoder.py
import torch
import torch.nn as nn
import math
import sys
sys.path.append('/kaggle/working/models')
from skateformer_block import STBlock


class SkateFormerEncoder(nn.Module):
    """
    SkateFormer Encoder - MEMORY OPTIMIZED
    """
    def __init__(self, num_coords=2, num_joints=17, n_frames=3, 
                 embed_dim=128, depth=2, num_heads=4, mlp_ratio=2.,
                 drop_rate=0., attn_drop_rate=0.):
        super().__init__()
        self.num_coords = num_coords
        self.num_joints = num_joints
        self.n_frames = n_frames
        self.embed_dim = embed_dim
        
        # Simplified input embedding
        self.input_embed = nn.Conv2d(num_coords, embed_dim, 
                                     kernel_size=1, stride=1, bias=False)
        
        # Learnable positional embedding
        self.pos_embed = nn.Parameter(
            torch.zeros(1, embed_dim, n_frames, num_joints)
        )
        
        # Reduced ST blocks
        self.blocks = nn.ModuleList([
            STBlock(
                dim=embed_dim, num_heads=num_heads, num_joints=num_joints,
                mlp_ratio=mlp_ratio, qkv_bias=False, drop=drop_rate,
                attn_drop=attn_drop_rate
            )
            for _ in range(depth)
        ])
        
        # Efficient pooling
        self.norm = nn.LayerNorm(embed_dim)
        
        # Latent projection
        self.latent_proj = nn.Linear(embed_dim, 128)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(128, embed_dim),
            nn.GELU(),
        )
        self.output_proj = nn.Conv2d(embed_dim, num_coords, 
                                     kernel_size=1, stride=1, bias=False)
        
        # Initialize
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        """
        x: [B, C, T, V]
        Returns: latent [B, 128], recon [B, C, T, V]
        """
        B, C, T, V = x.shape
        
        # Embed
        x = self.input_embed(x)  # [B, embed_dim, T, V]
        x = x + self.pos_embed
        
        # Apply blocks
        for block in self.blocks:
            x = block(x)
        
        # Extract features
        # [B, embed_dim, T, V] -> [B, T, V, embed_dim]
        x_norm = x.permute(0, 2, 3, 1).contiguous()
        x_norm = self.norm(x_norm)
        
        # Global average pooling
        latent_feat = x_norm.mean(dim=[1, 2])  # [B, embed_dim]
        latent = self.latent_proj(latent_feat)  # [B, 128]
        
        # Decode
        dec_feat = self.decoder(latent)  # [B, embed_dim]
        dec_feat = dec_feat.view(B, self.embed_dim, 1, 1)
        dec_feat = dec_feat.expand(-1, -1, T, V)
        
        # Residual connection
        dec_feat = dec_feat + x
        
        # Output
        recon = self.output_proj(dec_feat)  # [B, C, T, V]
        
        return latent, recon

print("✅ Memory-Efficient SkateFormer Encoder created!")

Overwriting /kaggle/working/models/skateformer_encoder.py


**Update SkateFormer Diffusion (Memory Efficient)**

In [27]:
%%writefile /kaggle/working/models/skateformer_diffusion.py
import torch
import torch.nn as nn
import math
import sys
sys.path.append('/kaggle/working/models')
from skateformer_block import STBlock


def timestep_embedding(timesteps, dim, max_period=10000):
    """Sinusoidal timestep embeddings"""
    half = dim // 2
    freqs = torch.exp(
        -math.log(max_period) * torch.arange(start=0, end=half, dtype=torch.float32) / half
    ).to(device=timesteps.device)
    args = timesteps[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding


class SkateFormerDiffusion(nn.Module):
    """
    SkateFormer Diffusion - MEMORY OPTIMIZED
    """
    def __init__(self, input_feats, num_frames=7, latent_dim=256,
                 depth=3, num_heads=4, num_joints=17, dropout=0.1,
                 inject_condition=True):
        super().__init__()
        
        self.input_feats = input_feats
        self.num_frames = num_frames
        self.latent_dim = latent_dim
        self.num_joints = num_joints
        self.inject_condition = inject_condition
        
        # Time + Condition embedding
        self.time_embed = nn.Sequential(
            nn.Linear(latent_dim, latent_dim // 2),
            nn.SiLU(),
            nn.Linear(latent_dim // 2, latent_dim // 2),
        )
        
        if inject_condition:
            self.cond_embed = nn.Linear(128, latent_dim // 2)
        
        # Input projection
        self.input_proj = nn.Linear(input_feats, latent_dim)
        
        # Reduced ST blocks
        self.blocks = nn.ModuleList([
            STBlock(
                dim=latent_dim, num_heads=num_heads, num_joints=num_joints,
                mlp_ratio=2., qkv_bias=False, drop=dropout,
                attn_drop=dropout
            )
            for _ in range(depth)
        ])
        
        # Time injection (shared across blocks to save memory)
        self.time_mlp = nn.Sequential(
            nn.Linear(latent_dim // 2, latent_dim),
            nn.SiLU(),
        )
        
        # Output
        self.norm_out = nn.LayerNorm(latent_dim)
        self.out_proj = nn.Linear(latent_dim, input_feats)
        
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def forward(self, x, timesteps, condition_data=None):
        """
        x: [B, T, D]
        timesteps: [B]
        condition_data: [B, 128]
        """
        B, T, D = x.shape
        
        # Time embedding
        t_emb = self.time_embed(timestep_embedding(timesteps, self.latent_dim))
        
        # Merge condition
        if self.inject_condition and condition_data is not None:
            cond_emb = self.cond_embed(condition_data)
            emb = t_emb + cond_emb
        else:
            emb = t_emb
        
        # Project input
        h = self.input_proj(x)  # [B, T, latent_dim]
        
        # Add time embedding
        time_feat = self.time_mlp(emb).unsqueeze(1)  # [B, 1, latent_dim]
        h = h + time_feat
        
        # Reshape for ST blocks: [B, T, latent_dim] -> [B, latent_dim, T, V]
        # Split D into spatial dimension
        h = h.view(B, T, self.num_joints, -1)  # [B, T, V, C]
        h = h.permute(0, 3, 1, 2).contiguous()  # [B, C, T, V]
        
        # Apply ST blocks
        for block in self.blocks:
            h = block(h)
        
        # Back to sequence: [B, C, T, V] -> [B, T, D]
        h = h.permute(0, 2, 3, 1).contiguous()  # [B, T, V, C]
        h = h.view(B, T, -1)  # [B, T, D]
        
        # Output
        h = self.norm_out(h)
        output = self.out_proj(h)
        
        return output, None, None, None

print("✅ Memory-Efficient SkateFormer Diffusion created!")

Overwriting /kaggle/working/models/skateformer_diffusion.py


**** Update Training Script****

In [28]:
%%writefile /kaggle/working/train_skateformer.py
import argparse
import os
import random
import numpy as np
import pytorch_lightning as pl
import torch
import yaml
import gc

import sys
sys.path.append('/kaggle/working/models')
from dcmd_skateformer import DCMD_SkateFormer

from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.strategies import DDPStrategy
from utils.argparser import init_args
from utils.dataset import get_dataset_and_loader

if __name__ == '__main__':
    # Clear cache
    torch.cuda.empty_cache()
    gc.collect()
    
    parser = argparse.ArgumentParser(description='SkateFormer_DCMD')
    parser.add_argument('-c', '--config', type=str, required=True)
    
    args = parser.parse_args()
    config_path = args.config
    args = yaml.load(open(args.config), Loader=yaml.FullLoader)
    args = argparse.Namespace(**args)
    args = init_args(args)
    
    os.system(f'cp {config_path} {os.path.join(args.ckpt_dir, "config.yaml")}')
    
    # Seeds
    torch.manual_seed(args.seed)
    random.seed(args.seed)
    np.random.seed(args.seed)
    pl.seed_everything(args.seed)
    
    print("✅ Seed set to", args.seed)
    
    # Callbacks
    callbacks = [
        ModelCheckpoint(
            dirpath=args.ckpt_dir,
            save_top_k=1,
            save_last=True,
            monitor='train_loss',
            mode='min',
            every_n_epochs=1
        )
    ]
    
    print("✅ Callbacks configured")
    
    # Data
    print("📊 Loading data...")
    _, train_loader, _, val_loader = get_dataset_and_loader(
        args, split=args.split, validation=args.validation
    )
    print(f"✅ Data loaded: {len(train_loader)} batches")
    
    # Model
    print("🔧 Building model...")
    model = DCMD_SkateFormer(args)
    
    # Trainer - MEMORY OPTIMIZED
    trainer = pl.Trainer(
        accelerator=args.accelerator,
        devices=args.devices,
        default_root_dir=args.ckpt_dir,
        max_epochs=args.n_epochs,
        logger=False,
        callbacks=callbacks,
        strategy='auto',
        log_every_n_steps=10,
        num_sanity_val_steps=0,
        deterministic=False,  # Faster
        gradient_clip_val=0.5,  # Prevent explosion
        precision='16-mixed',  # Mixed precision
        accumulate_grad_batches=4,  # Gradient accumulation
        enable_progress_bar=True,
        enable_checkpointing=True,
    )
    
    print("🏋️ Starting training...")
    
    # Train
    trainer.fit(model=model, train_dataloaders=train_loader)
    
    print("✅ Training completed!")
    
    # Clear cache
    torch.cuda.empty_cache()
    gc.collect()

Overwriting /kaggle/working/train_skateformer.py


**Update DCMD SkateFormer (Gradient Checkpointing)**

In [29]:
%%writefile /kaggle/working/models/dcmd_skateformer.py
import argparse
import os
from math import prod
from typing import Dict, List, Tuple
import numpy as np
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import torch.utils.checkpoint as checkpoint

import sys
sys.path.append('/kaggle/working/models')
from skateformer_encoder import SkateFormerEncoder
from skateformer_diffusion import SkateFormerDiffusion

sys.path.append('/kaggle/working/DCMD-main')
from utils.diffusion_utils import Diffusion
from utils.eval_utils import compute_var_matrix, filter_vectors_by_cond
from utils.model_utils import processing_data
from utils.tools import get_dct_matrix, generate_pad, padding_traj


class DCMD_SkateFormer(pl.LightningModule):
    """
    DCMD + SkateFormer - MEMORY OPTIMIZED
    """
    
    losses = {'l1': nn.L1Loss, 'smooth_l1': nn.SmoothL1Loss, 'mse': nn.MSELoss}
    
    def __init__(self, args: argparse.Namespace) -> None:
        super(DCMD_SkateFormer, self).__init__()
        
        self.save_hyperparameters(args)
        
        # Parameters
        self.n_frames = args.seg_len
        self.num_coords = args.num_coords
        self.n_joints = 17
        self.n_his = args.n_his
        
        # Training
        self.learning_rate = args.opt_lr
        self.loss_fn = self.losses[args.loss_fn](reduction='none')
        self.noise_steps = args.noise_steps
        self.aggregation_strategy = args.aggregation_strategy
        self.n_generated_samples = args.n_generated_samples
        self.model_return_value = args.model_return_value
        
        # Others
        self.split = args.split
        self.ckpt_dir = args.ckpt_dir
        self.save_tensors = args.save_tensors
        self.padding = args.padding
        
        self.idx_pad, self.zero_index = generate_pad(
            self.padding, self.n_his, self.n_frames - self.n_his
        )
        
        # Diffusion
        self._set_diffusion_variables()
        
        # Build models
        self.build_model(args)
    
    def build_model(self, args) -> None:
        """Build models"""
        
        print("🔨 Building SkateFormer models...")
        
        # Get parameters with defaults
        embed_dim = getattr(args, 'embed_dim', 128)
        encoder_depth = getattr(args, 'encoder_depth', 2)
        diffusion_depth = getattr(args, 'diffusion_depth', 3)
        num_heads = getattr(args, 'num_heads', 4)
        latent_dims = getattr(args, 'latent_dims', 256)
        
        # Encoder
        self.rec_model = SkateFormerEncoder(
            num_coords=self.num_coords,
            num_joints=self.n_joints,
            n_frames=self.n_his,
            embed_dim=embed_dim,
            depth=encoder_depth,
            num_heads=num_heads,
            mlp_ratio=2.,
            drop_rate=0.1,
            attn_drop_rate=0.1,
        )
        
        # Diffusion
        self.pre_model = SkateFormerDiffusion(
            input_feats=2 * self.n_joints,
            num_frames=self.n_frames,
            latent_dim=latent_dims,
            depth=diffusion_depth,
            num_heads=num_heads,
            num_joints=self.n_joints,
            dropout=0.1,
            inject_condition=True,
        )
        
        print("✅ Models built successfully!")
    
    def forward(self, input_data: List[torch.Tensor], aggr_strategy: str = None, 
                return_: str = None) -> List[torch.Tensor]:
        """Forward pass - SAME AS BEFORE"""
        
        tensor_data, meta_out = self._unpack_data(input_data)
        B = tensor_data.shape[0]
        
        history_data = tensor_data[:, :, :self.n_his, :]
        x_0 = padding_traj(history_data, self.padding, self.idx_pad, self.zero_index)
        
        generated_xs = []
        
        for _ in range(self.n_generated_samples):
            # Reconstruction
            with torch.cuda.amp.autocast():
                condition_embedding, rec_his_data = self.rec_model(history_data)
            
            # DCT
            dct_m, idct_m = get_dct_matrix(self.n_frames)
            dct_m_all = dct_m.float().to(self.device)
            idct_m_all = idct_m.float().to(self.device)
            
            x = x_0.permute(0, 2, 3, 1).contiguous()
            x = x.reshape([x.shape[0], self.n_frames, -1])
            y = torch.matmul(dct_m_all, x)
            
            # Diffusion
            y_d = torch.randn_like(y, device=self.device)
            
            for i in reversed(range(1, self.noise_steps)):
                t = torch.full(size=(B,), fill_value=i, dtype=torch.long, device=self.device)
                t_prev = torch.full(size=(B,), fill_value=i-1, dtype=torch.long, device=self.device)
                
                noise_pre = torch.randn_like(y_d) if i > 1 else torch.zeros_like(y_d)
                
                # Predict noise
                with torch.cuda.amp.autocast():
                    predicted_noise_pre, _, _, _ = self.pre_model(
                        y_d, t, condition_data=condition_embedding
                    )
                
                alpha_pre = self._alpha[t][:, None, None]
                alpha_hat_pre = self._alpha_hat[t][:, None, None]
                beta_pre = self._beta[t][:, None, None]
                
                y_d = (1 / torch.sqrt(alpha_pre)) * \
                      (y_d - ((1 - alpha_pre) / (torch.sqrt(1 - alpha_hat_pre))) * predicted_noise_pre) + \
                      torch.sqrt(beta_pre) * noise_pre
                
                alpha_hat_prev = self._alpha_hat[t_prev][:, None, None]
                y_n = (torch.sqrt(alpha_hat_prev) * y) + (torch.sqrt(1 - alpha_hat_prev) * noise_pre)
                
                # Mask
                mask = torch.zeros_like(x, device=self.device)
                mask[:, :self.n_his, :] = 1
                
                y_d_idct = torch.matmul(idct_m_all, y_d)
                y_n_idct = torch.matmul(idct_m_all, y_n)
                m_y = torch.mul((1-mask), y_d_idct) + torch.mul(mask, y_n_idct)
                y_d = torch.matmul(dct_m_all, m_y)
            
            # iDCT
            pre_future_data = torch.matmul(idct_m_all, y_d)
            pre_future_data = pre_future_data.reshape(B, self.n_frames, -1, 2)
            pre_future_data = pre_future_data.permute(0, 3, 1, 2).contiguous()
            pre_future_data = pre_future_data[:, :, self.n_his:, :]
            
            xs = torch.cat((rec_his_data, pre_future_data), dim=2)
            generated_xs.append(xs)
        
        selected_x, loss_of_selected_x = self._aggregation_strategy(
            generated_xs, tensor_data, aggr_strategy
        )
        
        return self._pack_out_data(selected_x, loss_of_selected_x, 
                                   [tensor_data] + meta_out, return_=return_)
    
    def training_step(self, batch: List[torch.Tensor], batch_idx: int) -> torch.float32:
        """Training step - SIMPLIFIED"""
        
        tensor_data, _ = self._unpack_data(batch)
        
        history_data = tensor_data[:, :, :self.n_his, :]
        x_0 = tensor_data
        
        # Reconstruction
        condition_embedding, rec_his_data = self.rec_model(history_data)
        rec_loss = torch.mean(self.loss_fn(rec_his_data, history_data))
        
        # Prediction
        dct_m, _ = get_dct_matrix(self.n_frames)
        dct_m_all = dct_m.float().to(self.device)
        x = x_0.permute(0, 2, 3, 1).contiguous()
        x = x.reshape([x.shape[0], self.n_frames, -1])
        y_0 = torch.matmul(dct_m_all, x)
        
        t = self.noise_scheduler.sample_timesteps(y_0.shape[0]).to(self.device)
        y_t, pre_noise = self.noise_scheduler.noise_motion(y_0, t)
        
        pre_predicted_noise, _, _, _ = self.pre_model(
            y_t, t, condition_data=condition_embedding
        )
        
        pre_loss = torch.mean(self.loss_fn(pre_predicted_noise, pre_noise))
        
        # Total loss
        total_loss = rec_loss + pre_loss
        
        self.log('rec_loss', rec_loss, prog_bar=True)
        self.log('pre_loss', pre_loss, prog_bar=True)
        self.log('train_loss', total_loss, prog_bar=True)
        
        return total_loss
    
    def configure_optimizers(self) -> Dict:
        """Optimizer"""
        optimizer = Adam(self.parameters(), lr=self.learning_rate, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=self.trainer.max_epochs, eta_min=1e-6
        )
        return {'optimizer': optimizer, 'lr_scheduler': scheduler}
    
    # Helper methods
    def _set_diffusion_variables(self) -> None:
        self.noise_scheduler = Diffusion(
            noise_steps=self.noise_steps,
            n_joints=self.n_joints,
            device=self.device,
            time=self.n_frames
        )
        self._beta_ = self.noise_scheduler.schedule_noise()
        self._alpha_ = (1. - self._beta_)
        self._alpha_hat_ = torch.cumprod(self._alpha_, dim=0)
    
    def _unpack_data(self, x: torch.Tensor) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        tensor_data = x[0].to(self.device)
        meta_out = [x[1], x[2], x[3]]
        return tensor_data, meta_out
    
    def _aggregation_strategy(self, generated_xs, input_sequence, aggr_strategy):
        aggr_strategy = self.aggregation_strategy if aggr_strategy is None else aggr_strategy
        
        if aggr_strategy == 'random':
            return generated_xs[np.random.randint(len(generated_xs))], None
        
        B, repr_shape = input_sequence.shape[0], input_sequence.shape[1:]
        compute_loss = lambda x: torch.mean(
            self.loss_fn(x, input_sequence).reshape(-1, prod(repr_shape)), dim=-1
        )
        losses = [compute_loss(x) for x in generated_xs]
        
        if aggr_strategy == 'best':
            loss_of_selected_x = torch.full((B,), fill_value=1e10, device=self.device)
            selected_x = torch.zeros((B, *repr_shape)).to(self.device)
            
            for i in range(len(generated_xs)):
                mask = losses[i] < loss_of_selected_x
                loss_of_selected_x[mask] = losses[i][mask]
                selected_x[mask] = generated_xs[i][mask]
        else:
            selected_x = None
            loss_of_selected_x = torch.mean(torch.stack(losses), dim=0)
        
        return selected_x, loss_of_selected_x
    
    def _pack_out_data(self, selected_x, loss_of_selected_x, additional_out, return_):
        if return_ is None:
            return_ = self.model_return_value
        
        out = []
        if return_ == 'loss' or return_ == 'all':
            if loss_of_selected_x is not None:
                out.append(loss_of_selected_x)
        if return_ == 'poses' or return_ == 'all':
            if selected_x is not None:
                out.append(selected_x)
        
        return out + additional_out
    
    @property
    def _beta(self):
        return self._beta_.to(self.device)
    
    @property
    def _alpha(self):
        return self._alpha_.to(self.device)
    
    @property
    def _alpha_hat(self):
        return self._alpha_hat_.to(self.device)

print("✅ DCMD_SkateFormer model created!")

Overwriting /kaggle/working/models/dcmd_skateformer.py


In [30]:
# Clear all GPU memory
import torch
import gc

torch.cuda.empty_cache()
gc.collect()
print("✅ GPU Memory cleared!")

✅ GPU Memory cleared!


In [31]:
%%writefile /kaggle/working/config/Avenue/skateformer_train.yaml
### Experiment configuration với SkateFormer - MEMORY OPTIMIZED

## General settings
split: 'train'
debug: false
seed: 999
validation: false
use_hr: false

## Computational resources
accelerator: 'gpu'
devices: [0]

## Paths
dir_name: 'skateformer_train'
data_dir: '/kaggle/input/avenue/Avenue'
exp_dir: '/kaggle/working/checkpoints/HR-Avenue'
test_path: '/kaggle/input/avenue/Avenue/testing/test_frame_mask'
load_ckpt: 'best.ckpt'
create_experiment_dir: true

## WANDB configuration
use_wandb: false
project_name: "dcmd_skateformer"
wandb_entity: "your_entity"
group_name: "avenue_experiments"
use_ema: false

### Model configuration - REDUCED for memory efficiency

## SkateFormer parameters
dropout: 0.1
conditioning_strategy: 'inject'
h_dim: 128  # Reduced from 256
latent_dim: 128  # Reduced from 256
channels: [128, 64, 128]  # Reduced

## Diffusion configuration
noise_steps: 10

### Training configuration - OPTIMIZED
n_epochs: 8  # Reduced epochs
opt_lr: 0.001

## Loss configuration
loss_fn: 'smooth_l1'

### Inference configuration
n_generated_samples: 20  # Reduced from 30
model_return_value: 'loss'
aggregation_strategy: 'best'
filter_kernel_size: 30
frames_shift: 6
save_tensors: false  # Don't save to reduce memory
load_tensors: false

### Dataset configuration - MEMORY OPTIMIZED
dataset_choice: 'HR-Avenue'
seg_len: 7
vid_res: [640, 360]
batch_size: 512  # REDUCED from 4096 to avoid OOM
pad_size: 12

headless: false
hip_center: false
kp18_format: false
normalization_strategy: 'robust'
num_coords: 2
num_transform: 5
num_workers: 4  # Reduced workers
seg_stride: 1
seg_th: 0
start_offset: 0
symm_range: true
use_fitted_scaler: false

n_his: 3
padding: 'LastFrame'

## SkateFormer specific - REDUCED
encoder_depth: 2  # Reduced from 3
diffusion_depth: 3  # Reduced from 4
num_heads: 4  # Reduced from 8
embed_dim: 128  # Reduced from 256
latent_dims: 256  # Reduced from 512

Overwriting /kaggle/working/config/Avenue/skateformer_train.yaml


#  RUN TRAINING# 

In [32]:
# Clear memory
import torch
import gc
torch.cuda.empty_cache()
gc.collect()

print("🚀 Starting SkateFormer + DCMD Training with OPTIMIZED settings...")
!python /kaggle/working/train_skateformer.py --config /kaggle/working/config/Avenue/skateformer_train.yaml

🚀 Starting SkateFormer + DCMD Training with OPTIMIZED settings...
✅ SkateFormer Attention Module created!
✅ OPTIMIZED SkateFormer Block created!
✅ Memory-Efficient SkateFormer Encoder created!
✅ Memory-Efficient SkateFormer Diffusion created!
✅ DCMD_SkateFormer model created!
Experiment directories created in /kaggle/working/checkpoints/HR-Avenue/HR-Avenue/skateformer_train
✅ Seed set to 999
✅ Callbacks configured
📊 Loading data...

Loaded 2786 trajectories.

Removed short trajectories. Number of trajectories left: 1735.

Changed local trajectories's coordinate system to bounding_box_centre.

Normalised local trajectories using the robust normalisation strategy.
✅ Data loaded: 738 batches
🔧 Building model...
🔨 Building SkateFormer models...
✅ Models built successfully!
🏋️ Starting training...
/usr/local/lib/python3.11/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:751: Checkpoint directory /kaggle/working/checkpoints/HR-Avenue/HR-Avenue/skateformer_train exists and is no

# **Pip**

In [33]:
!pip install appdirs==1.4.4 docker-pycreds==0.4.0 gitdb==4.0.10 gitpython==3.1.32 joblib==1.3.1 numpy==1.25.2 pathtools==0.1.2 protobuf==4.23.4 scikit-learn==1.3.0 scipy==1.11.1 sentry-sdk==1.29.2 setproctitle==1.3.2 smmap==5.0.0 threadpoolctl==3.2.0 wandb==0.15.8
!pip install --upgrade pytorch-lightning pyyaml wandb pandas numpy matplotlib matplotlib-inline scikit-learn tqdm
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

  Using cached sentry_sdk-1.29.2-py2.py3-none-any.whl.metadata (8.8 kB)
  Using cached wandb-0.15.8-py3-none-any.whl.metadata (8.3 kB)
Using cached sentry_sdk-1.29.2-py2.py3-none-any.whl (215 kB)
Using cached wandb-0.15.8-py3-none-any.whl (2.1 MB)
  Attempting uninstall: sentry-sdk
    Found existing installation: sentry-sdk 2.39.0
    Uninstalling sentry-sdk-2.39.0:
      Successfully uninstalled sentry-sdk-2.39.0
  Attempting uninstall: wandb
    Found existing installation: wandb 0.22.1
    Uninstalling wandb-0.22.1:
      Successfully uninstalled wandb-0.22.1
  Using cached wandb-0.22.1-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (10 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 1.3 MB/s eta 0:00:00
  Using cached sentry_sdk-2.39.0-py2.py3-none-any.whl.metadata (10 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.3 MB/s eta 0:00:00
Using cached wandb-0.22.1-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (19.6 MB)
   ━━━

# **Replace code to run properly**

In [34]:
%%writefile /kaggle/working/DCMD-main/config/Avenue/dcmd_train.yaml 
### Experiment configuration

## General settings
split: 'train' # data split; choices ['train', 'test']
debug: false # if true, load only a few data samples
seed: 999
validation: false # use validation; only for UBnormal
use_hr: false # for validation and test on UBnormal

## Computational resources
accelerator: 'gpu'
devices: [0] # indices of cuda devices to use

## Paths
dir_name: 'train_experiment' # name of the directory of the current experiment
data_dir: '/kaggle/input/avenue/Avenue' # path to the data
exp_dir: '/kaggle/working/DCMD-main/checkpoints' # path to the directory that will contain the current experiment directory
test_path: '/kaggle/input/avenue/Avenue/testing/test_frame_mask' # path to the test data
load_ckpt: 'best.ckpt' # name of the checkpoint to load at inference time
create_experiment_dir: true

## WANDB configuration
use_wandb: false
project_name: "project_name"
wandb_entity: "entity_name"
group_name: "group_name"
use_ema: false

##############################


### Model's configuration

## U-Net's configuration
dropout: 0. # probability of dropout
conditioning_strategy: 'inject'
## Rec configuration
h_dim: 512 # dimension of the bottleneck at the end of the encoder of the conditioning network
latent_dim: 256 # dimension of the latent space of the conditioning encoder
channels: [512,256,512] # channels for the encoder

##############################


### Training's configuration

## Diffusion's configuration
noise_steps: 10 # how many diffusion steps to perform

### Optimizer and scheduler's configuration
n_epochs: 10
opt_lr: 0.001

## Losses' configuration
loss_fn: 'smooth_l1' # loss function; choices ['mse', 'l1', 'smooth_l1']

##############################


### Inference's configuration
n_generated_samples: 50 # number of samples to generate
model_return_value: 'loss' # choices ['loss', 'poses', 'all']; if 'loss', the model will return the loss;
                           # if 'poses', the model will return the generated poses; 
                           # if 'all', the model will return both the loss and the generated poses
aggregation_strategy: 'best' # choices ['best', 'mean', 'median', 'random']; if 'best', the best sample will be selected; 
                             # if 'mean', the mean of loss of the samples will be selected; 
                             # if 'median', the median of the loss of the samples will be selected; 
                             # if 'random', a random sample will be selected;
                             # if 'mean_poses', the mean of the generated poses will be selected;
                             # if 'median_poses', the median of the generated poses will be selected;
                             # if 'all', all the generated poses will be selected
filter_kernel_size: 30 # size of the kernel to use for smoothing the anomaly score of each clip
frames_shift: 6 # it compensates the shift of the anomaly score due to the sliding window; 
                # in conjuction with pad_size and filter_kernel_size, it strongly depends on the dataset
save_tensors: false # if true, save the generated tensors for faster inference
load_tensors: false # if true, load the generated tensors for faster inference

##############################


### Dataset's configuration

## Important parameters
dataset_choice: 'HR-Avenue'
seg_len: 7 # length of the window (his+pre)
vid_res: [640,360]
batch_size: 2048
pad_size: 12 # size of the padding 

## Other parameters
headless: false # remove the keypoints of the head
hip_center: false # center the keypoints on the hip
kp18_format: false # use the 18 keypoints format
normalization_strategy: 'robust' # use 'none' to avoid normalization, 'robust' otherwise
num_coords: 2 # number of coordinates to use
num_transform: 5 # number of transformations to apply
num_workers: 4
seg_stride: 1
seg_th: 0
start_offset: 0
symm_range: true
use_fitted_scaler: false

## New configuration
n_his: 3
padding: 'LastFrame'
## translinear configuration
num_layers: 6
num_heads: 8
latent_dims: 512
loss_1_series_weight: 0.01
loss_1_prior_weight: 0
loss_2_series_weight: 0
loss_2_prior_weight: 0.01

Overwriting /kaggle/working/DCMD-main/config/Avenue/dcmd_train.yaml


In [35]:
%%writefile /kaggle/working/DCMD-main/models/transformer.py
import torch
import torch.nn.functional as F
from torch import layer_norm, nn
import numpy as np
from typing import List, Tuple, Union

import math


def timestep_embedding(timesteps, dim, max_period=10000):
    """
    Create sinusoidal timestep embeddings.
    :param timesteps: a 1-D Tensor of N indices, one per batch element.
                      These may be fractional.
    :param dim: the dimension of the output.
    :param max_period: controls the minimum frequency of the embeddings.
    :return: an [N x dim] Tensor of positional embeddings.
    """
    half = dim // 2
    freqs = torch.exp(
        -math.log(max_period) * torch.arange(start=0, end=half, dtype=torch.float32) / half
    ).to(device=timesteps.device)
    args = timesteps[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding


def set_requires_grad(nets, requires_grad=False):
    """Set requies_grad for all the networks.

    Args:
        nets (nn.Module | list[nn.Module]): A list of networks or a single
            network.
        requires_grad (bool): Whether the networks require gradients or not
    """
    if not isinstance(nets, list):
        nets = [nets]
    for net in nets:
        if net is not None:
            for param in net.parameters():
                param.requires_grad = requires_grad


def zero_module(module):
    """
    Zero out the parameters of a module and return it.
    """
    for p in module.parameters():
        p.detach().zero_()
    return module


class StylizationBlock(nn.Module):

    def __init__(self, latent_dim, time_embed_dim, dropout):
        super().__init__()
        self.emb_layers = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_embed_dim, 2 * latent_dim),
        )
        self.norm = nn.LayerNorm(latent_dim)
        self.out_layers = nn.Sequential(
            nn.SiLU(),
            nn.Dropout(p=dropout),
            zero_module(nn.Linear(latent_dim, latent_dim)),
        )

    def forward(self, h, emb):
        """
        h: B, T, D
        emb: B, D
        """
        # B, 1, 2D
        emb_out = self.emb_layers(emb).unsqueeze(1)
        # scale: B, 1, D / shift: B, 1, D
        scale, shift = torch.chunk(emb_out, 2, dim=2)
        # B, T, D
        h = self.norm(h) * (1 + scale) + shift
        h = self.out_layers(h)
        return h


class FFN(nn.Module):

    def __init__(self, latent_dim, ffn_dim, dropout, time_embed_dim):
        super().__init__()
        self.linear1 = nn.Linear(latent_dim, ffn_dim)
        self.linear2 = zero_module(nn.Linear(ffn_dim, latent_dim))
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.proj_out = StylizationBlock(latent_dim, time_embed_dim, dropout)

    def forward(self, x, emb):
        """
            x: B, T, D (D=latent_dim)
        """
        y = self.linear2(self.dropout(self.activation(self.linear1(x))))
        y = x + self.proj_out(y, emb)
        return y


class TemporalSelfAttention(nn.Module):

    def __init__(self, n_frames, latent_dim, num_head, dropout, time_embed_dim, output_attention = True):
        super().__init__()
        self.num_head = num_head
        self.output_attention = output_attention
        self.norm = nn.LayerNorm(latent_dim)
        self.query = nn.Linear(latent_dim, latent_dim, bias=False)
        self.key = nn.Linear(latent_dim, latent_dim, bias=False)
        self.value = nn.Linear(latent_dim, latent_dim, bias=False)
        self.sigma_projection = nn.Linear(latent_dim, num_head, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.proj_out = StylizationBlock(latent_dim, time_embed_dim, dropout)
        n_frames = n_frames
        self.distances = torch.zeros((n_frames, n_frames)).cuda(0)

        for i in range(n_frames):
            for j in range(n_frames):
                self.distances[i][j] = abs(i - j)

    def forward(self, x, emb):
        """
        x: B, T, D (D=latent_dim)
        """
        B, T, D = x.shape
        H = self.num_head

        ## series-association
        # B, T, 1, D
        query = self.query(self.norm(x)).unsqueeze(2)
        # B, 1, T, D
        key = self.key(self.norm(x)).unsqueeze(1)
        # B, T, H, D/H
        query = query.view(B, T, H, -1)
        key = key.view(B, T, H, -1)
        scale = 1. / math.sqrt(D/H)
        # B, H, T, T
        scores = torch.einsum('bnhd,bmhd->bhnm', query, key) / math.sqrt(D // H)
        attention = scale * scores
        # B, H, T, T
        series = self.dropout(F.softmax(attention, dim=-1))

        ## prior-association
        sigma = self.sigma_projection(x).view(B, T, H)  # B, T, H
        sigma = sigma.transpose(1, 2)  # B T H ->  B H T
        sigma = torch.sigmoid(sigma * 5) + 1e-5
        sigma = torch.pow(3, sigma) - 1
        sigma = sigma.unsqueeze(-1).repeat(1, 1, 1, T)  # B, H, T, T
        prior = self.distances.unsqueeze(0).unsqueeze(0).repeat(sigma.shape[0], sigma.shape[1], 1, 1) # B, H, T, T
        prior = 1.0 / (math.sqrt(2 * math.pi) * sigma) * torch.exp(-prior ** 2 / 2 / (sigma ** 2)).cuda(0) # B, H, T, T

        # B, T, H, D/H
        value = self.value(self.norm(x)).view(B, T, H, -1)
        # B, T, D
        y = torch.einsum('bhnm,bmhd->bnhd', series, value).reshape(B, T, D)
        y = x + self.proj_out(y, emb)

        if self.output_attention:
            return y.contiguous(), series, prior, sigma
        else:
            return y.contiguous(), None

class TemporalDiffusionTransformerDecoderLayer(nn.Module):

    def __init__(self,
                 n_frames = 7,
                 latent_dim=16,
                 time_embed_dim=16,
                 ffn_dim=32,
                 num_head=4,
                 dropout=0.5
                 ):
        super().__init__()
        self.sa_block = TemporalSelfAttention(
            n_frames, latent_dim, num_head, dropout, time_embed_dim)
        self.ffn = FFN(latent_dim, ffn_dim, dropout, time_embed_dim)

    def forward(self, x, emb):
        x, series, prior, sigma = self.sa_block(x, emb)
        x = self.ffn(x, emb)
        return x, series, prior, sigma


class MotionTransformer(nn.Module):
    def __init__(self,
                 input_feats,
                 num_frames=7,
                 latent_dim=16,
                 ff_size=32,
                 num_layers=8,
                 num_heads=8,
                 dropout=0.2,
                 activation="gelu",
                 output_attention = True,
                 device: Union[str, torch.DeviceObjType] = 'cpu',
                 inject_condition: bool = False,
                 **kargs):
        super().__init__()


        self.input_feats = input_feats # 34
        self.num_frames = num_frames
        self.latent_dim = latent_dim
        self.ff_size = ff_size
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.dropout = dropout
        self.activation = activation
        self.output_attention = output_attention
        self.device = device
        self.time_embed_dim = latent_dim
        self.inject_condition = inject_condition

        self.build_model()

    def build_model(self):
        self.sequence_embedding = nn.Parameter(torch.randn(self.num_frames, self.latent_dim))

        # Input Embedding
        self.joint_embed = nn.Linear(self.input_feats, self.latent_dim)
        self.cond_embed = nn.Linear(256, self.time_embed_dim)

        self.time_embed = nn.Sequential(
            nn.Linear(self.latent_dim, self.time_embed_dim),
            nn.SiLU(),
            nn.Linear(self.time_embed_dim, self.time_embed_dim),
        )

        self.temporal_decoder_blocks = nn.ModuleList()
        for i in range(self.num_layers):
            self.temporal_decoder_blocks.append(
                TemporalDiffusionTransformerDecoderLayer(
                    n_frames=self.num_frames,
                    latent_dim=self.latent_dim,
                    time_embed_dim=self.time_embed_dim,
                    ffn_dim=self.ff_size,
                    num_head=self.num_heads,
                    dropout=self.dropout,
                )
            )
        # Output Module
        self.out = zero_module(nn.Linear(self.latent_dim, self.input_feats))


    def forward(self, x, timesteps, condition_data:torch.Tensor=None):
        """
        x: B, T, D (D=C*V)
        """
        B, T = x.shape[0], x.shape[1]

        # B, latent_dim
        emb = self.time_embed(timestep_embedding(timesteps, self.latent_dim))

        # Add conditioning signal
        if self.inject_condition:
            condition_data = self.cond_embed(condition_data)
            emb = emb + condition_data

        # B, T, latent_dim
        h = self.joint_embed(x)
        h = h + self.sequence_embedding.unsqueeze(0)[:, :T, :]
        
        i = 0
        prelist = []
        series_list = []
        prior_list = []
        sigma_list = []
        for module in self.temporal_decoder_blocks:
            if i < (self.num_layers // 2):
                prelist.append(h)
                h, series, prior, sigmas = module(h, emb) # B, T, latent_dim
                series_list.append(series)
                prior_list.append(prior)
                sigma_list.append(sigmas)
            elif i >= (self.num_layers // 2):
                h, series, prior, sigmas = module(h, emb)
                h += prelist[-1]
                series_list.append(series)
                prior_list.append(prior)
                sigma_list.append(sigmas)
                prelist.pop()
            i += 1

        # B, T, C*V
        output = self.out(h).view(B, T, -1).contiguous()
        if self.output_attention:
            return output, series_list, prior_list, sigma_list
        return output

Overwriting /kaggle/working/DCMD-main/models/transformer.py


In [36]:
%%writefile /kaggle/working/DCMD-main/train_DCMD.py
import argparse
import os
import random

import numpy as np
import pytorch_lightning as pl
import torch
import yaml
from models.dcmd import DCMD
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.strategies import DDPStrategy
from utils.argparser import init_args
from utils.dataset import get_dataset_and_loader
from utils.ema import EMACallback


if __name__== '__main__':

    # Parse command line arguments and load config file
    parser = argparse.ArgumentParser(description='Pose_AD_Experiment')
    parser.add_argument('-c', '--config', type=str, required=True,
                        default='/your_default_config_file_path')
    
    args = parser.parse_args()
    config_path = args.config
    args = yaml.load(open(args.config), Loader=yaml.FullLoader)
    args = argparse.Namespace(**args)
    args = init_args(args) 
    # Save config file to ckpt_dir
    os.system(f'cp {config_path} {os.path.join(args.ckpt_dir, "config.yaml")}')     
    
    # Set seeds    
    torch.manual_seed(args.seed)
    random.seed(args.seed)
    np.random.seed(args.seed) 
    pl.seed_everything(args.seed)

    # Set callbacks and logger
    if (hasattr(args, 'diffusion_on_latent') and args.stage == 'pretrain'):
        monitored_metric = 'pretrain_rec_loss'
        metric_mode = 'min'
    elif args.validation:
        monitored_metric = 'AUC'
        metric_mode = 'max'
    else:
        monitored_metric = 'loss'
        metric_mode = 'min'
    callbacks = [ModelCheckpoint(
                    dirpath=args.ckpt_dir,
                    save_top_k=2,
                    save_last=True,
                    monitor=monitored_metric,
                    mode=metric_mode
                )]

    
    callbacks += [EMACallback()] if args.use_ema else [] # Use to achieve exponential moving average
    
    if args.use_wandb:
        callbacks += [LearningRateMonitor(logging_interval='step')]
        wandb_logger = WandbLogger(project=args.project_name, group=args.group_name, entity=args.wandb_entity, 
                                   name=args.dir_name, config=vars(args), log_model='all')
    else:
        wandb_logger = False

    # Get dataset and loaders
    _, train_loader, _, val_loader = get_dataset_and_loader(args, split=args.split, validation=args.validation)
    
    # Initialize model and trainer
    model = DCMD(args)
    
    trainer = pl.Trainer(accelerator=args.accelerator, devices=args.devices, default_root_dir=args.ckpt_dir, max_epochs=args.n_epochs, 
                         logger=wandb_logger, callbacks=callbacks, strategy=DDPStrategy(find_unused_parameters=False),
                         log_every_n_steps=20, num_sanity_val_steps=0, deterministic=True)
    
    # Train the model    
    trainer.fit(model=model, train_dataloaders=train_loader, val_dataloaders=val_loader)

Overwriting /kaggle/working/DCMD-main/train_DCMD.py


In [37]:
%%writefile /kaggle/working/DCMD-main/config/Avenue/dcmd_test.yaml
### Experiment configuration

## General settings
split: 'test' # data split; choices ['train', 'test']
debug: false # if true, load only a few data samples
seed: 999
validation: false # use validation; only for UBnormal
use_hr: false # for validation and test on UBnormal

## Computational resources
accelerator: 'gpu'
devices: [0] # indices of cuda devices to use

## Paths
dir_name: 'test_experiment' # name of the directory of the current experiment
data_dir: '/kaggle/input/avenue/Avenue' # path to the data
exp_dir: '/kaggle/working/DCMD-main/checkpoints' # path to the directory that will contain the current experiment directory
test_path: '/kaggle/input/avenue/Avenue/testing/test_frame_mask' # path to the test data
load_ckpt: 'last.ckpt' # name of the checkpoint to load at inference time
create_experiment_dir: false # if true, create a new directory for the current experiment

## WANDB configuration
use_wandb: false
project_name: "project_name"
wandb_entity: "entity_name"
group_name: "group_name"
use_ema: false

##############################


### Model's configuration

## U-Net's configuration
dropout: 0. # probability of dropout
conditioning_strategy: 'inject'
## Rec configuration
h_dim: 512 # dimension of the bottleneck at the end of the encoder of the conditioning network
latent_dim: 256 # dimension of the latent space of the conditioning encoder
channels: [512,256,512] # channels for the encoder

##############################


### Training's configuration

## Diffusion's configuration
noise_steps: 10 # how many diffusion steps to perform

### Optimizer and scheduler's configuration
n_epochs: 10
opt_lr: 0.001

## Losses' configuration
loss_fn: 'smooth_l1' # loss function; choices ['mse', 'l1', 'smooth_l1']

##############################


### Inference's configuration
n_generated_samples: 50 # number of samples to generate
model_return_value: 'loss' # choices ['loss', 'poses', 'all']; if 'loss', the model will return the loss;
                           # if 'poses', the model will return the generated poses; 
                           # if 'all', the model will return both the loss and the generated poses
aggregation_strategy: 'best' # choices ['best', 'mean', 'median', 'random']; if 'best', the best sample will be selected; 
                             # if 'mean', the mean of loss of the samples will be selected; 
                             # if 'median', the median of the loss of the samples will be selected; 
                             # if 'random', a random sample will be selected;
                             # if 'mean_poses', the mean of the generated poses will be selected;
                             # if 'median_poses', the median of the generated poses will be selected;
                             # if 'all', all the generated poses will be selected
filter_kernel_size: 30 # size of the kernel to use for smoothing the anomaly score of each clip
frames_shift: 6 # it compensates the shift of the anomaly score due to the sliding window; 
                # in conjuction with pad_size and filter_kernel_size, it strongly depends on the dataset
save_tensors: true # if true, save the generated tensors for faster inference
load_tensors: false # if true, load the generated tensors for faster inference

##############################


### Dataset's configuration

## Important parameters
dataset_choice: 'HR-Avenue'
seg_len: 7 # length of the window (cond+noised)
vid_res: [640,360]
batch_size: 2048
pad_size: 12 # size of the padding

## Other parameters
headless: false # remove the keypoints of the head
hip_center: false # center the keypoints on the hip
kp18_format: false # use the 18 keypoints format
normalization_strategy: 'robust' # use 'none' to avoid normalization, 'robust' otherwise
num_coords: 2 # number of coordinates to use
num_transform: 5 # number of transformations to apply
num_workers: 4
seg_stride: 1
seg_th: 0
start_offset: 0
symm_range: true
use_fitted_scaler: false

## New configuration
n_his: 3
padding: 'LastFrame'
## translinear configuration
num_layers: 6
num_heads: 8
latent_dims: 512
loss_1_series_weight: 0.01
loss_1_prior_weight: 0
loss_2_series_weight: 0
loss_2_prior_weight: 0.01

Overwriting /kaggle/working/DCMD-main/config/Avenue/dcmd_test.yaml


In [38]:
%%writefile /kaggle/working/DCMD-main/models/dcmd.py
import argparse
import os
from math import prod
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib import pyplot as plt

from models.stsae.stsae import STSAE
from sklearn.metrics import roc_curve, roc_auc_score
from torch.optim import Adam
from tqdm import tqdm

from utils.diffusion_utils import Diffusion
from utils.eval_utils import (compute_var_matrix, filter_vectors_by_cond,
                              get_avenue_mask, get_hr_ubnormal_mask, pad_scores, score_process)
from utils.model_utils import processing_data, my_kl_loss
from models.transformer import MotionTransformer
from utils.tools import get_dct_matrix, generate_pad, padding_traj


class DCMD(pl.LightningModule):

    losses = {'l1': nn.L1Loss, 'smooth_l1': nn.SmoothL1Loss, 'mse': nn.MSELoss}
    conditioning_strategies = {'inject': 'inject'}

    def __init__(self, args: argparse.Namespace) -> None:
        """
        This class implements DCMD model.

        Args:
            args (argparse.Namespace): arguments containing the hyperparameters of the model
        """

        super(DCMD, self).__init__()

        ## Log the hyperparameters of the model
        self.save_hyperparameters(args)

        ## Set the internal variables of the model
        # Data parameters
        self.n_frames = args.seg_len
        self.num_coords = args.num_coords
        self.n_joints = self._infer_number_of_joint(args)

        ## Model parameters
        # Main network
        self.dropout = args.dropout
        self.conditioning_strategy = self.conditioning_strategies[args.conditioning_strategy]
        # Conditioning network
        self.cond_h_dim = args.h_dim
        self.cond_latent_dim = args.latent_dim
        self.cond_channels = args.channels
        self.cond_dropout = args.dropout

        ## Training and inference parameters
        self.learning_rate = args.opt_lr
        self.loss_fn = self.losses[args.loss_fn](reduction='none')
        self.noise_steps = args.noise_steps
        self.aggregation_strategy = args.aggregation_strategy
        self.n_generated_samples = args.n_generated_samples
        self.model_return_value = args.model_return_value
        self.gt_path = args.gt_path
        self.split = args.split
        self.use_hr = args.use_hr
        self.ckpt_dir = args.ckpt_dir
        self.save_tensors = args.save_tensors
        self.num_transforms = args.num_transform
        self.anomaly_score_pad_size = args.pad_size
        self.anomaly_score_filter_kernel_size = args.filter_kernel_size
        self.anomaly_score_frames_shift = args.frames_shift
        self.dataset_name = args.dataset_choice

        # New parameters
        self.n_his = args.n_his
        self.padding = args.padding
        self.num_layers = args.num_layers
        self.num_heads = args.num_heads
        self.latent_dims = args.latent_dims
        self.automatic_optimization = False
        self.loss_1_series_weight = args.loss_1_series_weight
        self.loss_1_prior_weight = args.loss_1_prior_weight
        self.loss_2_series_weight = args.loss_2_series_weight
        self.loss_2_prior_weight = args.loss_2_prior_weight
        self.idx_pad, self.zero_index = generate_pad(self.padding, self.n_his, self.n_frames-self.n_his)

        ## Set the noise scheduler for the diffusion process
        self._set_diffusion_variables()

        ## Build the model
        self.build_model()


    def build_model(self) -> None:
        """
        Build the model according to the specified hyperparameters.
        """

        # Prediction Model
        pre_model = MotionTransformer(
            input_feats=2 * self.n_joints,
            num_frames=self.n_frames,
            num_layers=self.num_layers,
            num_heads=self.num_heads,
            latent_dim=self.latent_dims,
            dropout=self.dropout,
            device=self.device,
            inject_condition=(self.conditioning_strategy == 'inject')
        )

        # Reconstruction Model
        rec_model = STSAE(
            c_in=self.num_coords,
            h_dim=self.cond_h_dim,
            latent_dim=self.cond_latent_dim,
            n_frames=self.n_his,
            dropout=self.cond_dropout,
            n_joints=self.n_joints,
            layer_channels=self.cond_channels,
            device=self.device)

        self.pre_model, self.rec_model = pre_model, rec_model


    def forward(self, input_data: List[torch.Tensor], aggr_strategy: str = None, return_: str = None) -> List[torch.Tensor]:
        """
        Forward pass of the model.
        """

        ## Unpack data: tensor_data is the input data, meta_out is a list of metadata
        tensor_data, meta_out = self._unpack_data(input_data)
        B = tensor_data.shape[0]

        ## Select frames to reconstruct and to predict
        history_data = tensor_data[:, :, :self.n_his, :]
        x_0 = padding_traj(history_data, self.padding, self.idx_pad, self.zero_index)

        generated_xs = []
        # Generate m future predictions
        for _ in range(self.n_generated_samples):

            ## Reconstruction —— AE model
            condition_embedding, rec_his_data = self.rec_model(history_data)

            ## Prediction —— diffusion model
            ## DCT transformation
            dct_m, idct_m = get_dct_matrix(self.n_frames)
            dct_m_all = dct_m.float().to(self.device)
            idct_m_all = idct_m.float().to(self.device)
            # (B, C, T, V) -> (B, T, V, C)
            x = x_0.permute(0, 2, 3, 1).contiguous()
            # (B, T, V, C) -> (B, T, C*V)
            x = x.reshape([x.shape[0], self.n_frames, -1])
            y = torch.matmul(dct_m_all, x)  # [B, T, C*V]

            ## Generate gaussian noise of the same shape as the y
            y_d = torch.randn_like(y, device=self.device)

            ## (t ∈ T, T-1, ..., 1)
            for i in reversed(range(1, self.noise_steps)):

                ### Prediction (Two branches)
                ## Set the time step
                t = torch.full(size=(B,), fill_value=i, dtype=torch.long, device=self.device)
                t_prev = torch.full(size=(B,), fill_value=i, dtype=torch.long, device=self.device)
                t_prev[0] = 0

                ## Generate gaussian noise of the same shape as the predicted noise
                noise_pre = torch.randn_like(y_d, device=self.device) if i > 1 else torch.zeros_like(y_d, device=self.device)

                ## First branch
                # Predict the noise
                predicted_noise_pre, series, prior, _ = self.pre_model(y_d, t, condition_data=condition_embedding)
                # Get the alpha and beta values and expand them to the shape of the predicted noise
                alpha_pre = self._alpha[t][:, None, None]
                alpha_hat_pre = self._alpha_hat[t][:, None, None]
                beta_pre = self._beta[t][:, None, None]
                # Recover the predicted sequence
                y_d = (1 / torch.sqrt(alpha_pre)) * (y_d - ((1 - alpha_pre) / (torch.sqrt(1 - alpha_hat_pre))) * predicted_noise_pre) \
                    + torch.sqrt(beta_pre) * noise_pre
                ## Second branch
                alpha_hat_prev = self._alpha_hat[t_prev][:, None, None]
                # Add noise
                y_n = (torch.sqrt(alpha_hat_prev) * y) + (torch.sqrt(1 - alpha_hat_prev) * noise_pre)
                ## Mask completion
                # Get M values
                mask = torch.zeros_like(x, device=self.device) # [batch, T, C*V]
                for m in range(0, self.n_his):
                    mask[:, m, :] = 1
                # iDCT transformation
                y_d_idct = torch.matmul(idct_m_all, y_d)
                y_n_idct = torch.matmul(idct_m_all, y_n)
                # mask-mul
                m_mul_y_n = torch.mul(mask, y_n_idct)
                m_mul_y_d = torch.mul((1-mask), y_d_idct)
                # together
                m_y = m_mul_y_d + m_mul_y_n
                # DCT again
                y_d = torch.matmul(dct_m_all, m_y)

            # iDCT
            pre_future_data = torch.matmul(idct_m_all, y_d)
            # (B, T, C*V) -> (B, T, V, C)
            pre_future_data = pre_future_data.reshape(pre_future_data.shape[0], pre_future_data.shape[1], -1, 2)
            # (B, T, V, C) -> (B, C, T, V)
            pre_future_data = pre_future_data.permute(0, 3, 1, 2).contiguous()
            # select future sequences
            pre_future_data = pre_future_data[:,:,self.n_his:,:]

            ## Reconstruction + Prediction
            xs = torch.cat((rec_his_data, pre_future_data), dim=2)

            generated_xs.append(xs)

        selected_x, loss_of_selected_x = self._aggregation_strategy(generated_xs, tensor_data, aggr_strategy)

        return self._pack_out_data(selected_x, loss_of_selected_x, [tensor_data] + meta_out, return_=return_)


    def training_step(self, batch: List[torch.Tensor], batch_idx: int) -> torch.float32:
        """
        Training step of the model.
        """

        ## Get the optimizer returned in configuration_optimizers()
        opt = self.optimizers()

        ## Unpack data: tensor_data is the input data
        tensor_data, _ = self._unpack_data(batch)

        ## Select frames to reconstruct and to predict
        history_data = tensor_data[:, :, :self.n_his, :] # Used for rec (first n_his)
        x_0 = tensor_data # Used for pre（all）

        ## Reconstruction
        # Encode the history data
        condition_embedding, rec_his_data = self.rec_model(history_data)
        # Compute the rec_loss
        rec_loss = torch.mean(self.loss_fn(rec_his_data, history_data))
        self.log('rec_loss', rec_loss)

        ## Prediction
        # DCT transformation
        dct_m, _ = get_dct_matrix(self.n_frames)
        dct_m_all = dct_m.float().to(self.device)
        # (B, C, T, V) -> (B, T, V, C)
        x = x_0.permute(0, 2, 3, 1).contiguous()
        # (B, T, V, C) -> (B, T, C*V)
        x = x.reshape([x.shape[0], self.n_frames, -1]) # [batch, T, C*V]
        y_0 = torch.matmul(dct_m_all, x)

        # Sample the time steps and corrupt the data
        t = self.noise_scheduler.sample_timesteps(y_0.shape[0]).to(self.device)
        y_t, pre_noise = self.noise_scheduler.noise_motion(y_0, t) # (B, T, C*(V-1))

        # Predict the noise
        pre_predicted_noise, series, prior, _ = self.pre_model(y_t, t, condition_data=condition_embedding)

        # Compute the pre_loss
        # Calculate Association discrepancy
        series_loss = 0.0
        prior_loss = 0.0
        for u in range(len(prior)):
            # Pdetach, S <-> Maximize
            series_loss += \
                (torch.mean(my_kl_loss(
                    series[u],
                    (prior[u] / torch.unsqueeze(torch.sum(prior[u], dim=-1), dim=-1).repeat(1, 1, 1, self.n_frames)).detach()))
                + torch.mean(my_kl_loss(
                    (prior[u] / torch.unsqueeze(torch.sum(prior[u], dim=-1), dim=-1).repeat(1, 1, 1, self.n_frames)).detach(),
                    series[u])))
            # P, Sdetach <-> Minimize
            prior_loss += \
                (torch.mean(my_kl_loss(
                    (prior[u] / torch.unsqueeze(torch.sum(prior[u], dim=-1), dim=-1).repeat(1, 1, 1, self.n_frames)),
                    series[u].detach()))
                + torch.mean(my_kl_loss(
                    series[u].detach(),
                    (prior[u] / torch.unsqueeze(torch.sum(prior[u], dim=-1), dim=-1).repeat(1, 1, 1, self.n_frames)))))
        series_loss = series_loss / len(prior)
        prior_loss = prior_loss / len(prior)

        pre_loss = torch.mean(self.loss_fn(pre_predicted_noise, pre_noise))
        self.log('pre_loss', pre_loss)

        ## Compute loss1 & loss2
        loss1 = rec_loss + pre_loss \
                - self.loss_1_series_weight * series_loss \
                + self.loss_1_prior_weight * prior_loss
        self.log('loss1', loss1)
        loss2 = rec_loss + pre_loss \
                + self.loss_2_prior_weight * prior_loss \
                + self.loss_2_series_weight * series_loss
        self.log('loss2', loss2)

        ## Minimax strategy
        self.manual_backward(loss1, retain_graph=True)
        self.manual_backward(loss2)
        opt.step()
        opt.zero_grad()


    def test_step(self, batch: List[torch.Tensor], batch_idx: int) -> None:
        """
        Test step of the model. It saves the output of the model and the input data as
        List[torch.Tensor]: [predicted poses and the loss, tensor_data, transformation_idx, metadata, actual_frames]

        Args:
            batch (List[torch.Tensor]): list containing the following tensors:
                                         - tensor_data: tensor of shape (B, C, T, V) containing the input sequences
                                         - transformation_idx
                                         - metadata
                                         - actual_frames
            batch_idx (int): index of the batch
        """

        self._test_output_list.append(self.forward(batch))
        return


    def on_test_epoch_start(self) -> None:
        """
        Called when the test epoch begins.
        """

        super().on_test_epoch_start()
        self._test_output_list = []
        return


    def on_test_epoch_end(self) -> float:
        """
        Test epoch end of the model.

        Returns:
            float: test auc score
        """

        out, gt_data, trans, meta, frames = processing_data(self._test_output_list)
        del self._test_output_list
        if self.save_tensors:
            tensors = {'prediction': out, 'gt_data': gt_data,
                       'trans': trans, 'metadata': meta, 'frames': frames}
            self._save_tensors(tensors, split_name=self.split, aggr_strategy=self.aggregation_strategy, n_gen=self.n_generated_samples)
        auc_score = self.post_processing(out, gt_data, trans, meta, frames)
        self.log('AUC', auc_score)
        return auc_score


    def validation_step(self, batch: List[torch.Tensor], batch_idx: int) -> None:
        """
        Validation step of the model. It saves the output of the model and the input data as
        List[torch.Tensor]: [predicted poses and the loss, tensor_data, transformation_idx, metadata, actual_frames]

        Args:
            batch (List[torch.Tensor]): list containing the following tensors:
                                         - tensor_data: tensor of shape (B, C, T, V) containing the input sequences
                                         - transformation_idx
                                         - metadata
                                         - actual_frames
            batch_idx (int): index of the batch
        """

        self._validation_output_list.append(self.forward(batch))
        return


    def on_validation_epoch_start(self) -> None:
        """
        Called when the test epoch begins.
        """

        super().on_validation_epoch_start()
        self._validation_output_list = []
        return


    def on_validation_epoch_end(self) -> float:
        """
        Validation epoch end of the model.

        Returns:
            float: validation auc score
        """

        out, gt_data, trans, meta, frames = processing_data(self._validation_output_list)
        del self._validation_output_list
        if self.save_tensors:
            tensors = {'prediction': out, 'gt_data': gt_data,
                       'trans': trans, 'metadata': meta, 'frames': frames}
            self._save_tensors(tensors, split_name=self.split, aggr_strategy=self.aggregation_strategy, n_gen=self.n_generated_samples)
        auc_score = self.post_processing(out, gt_data, trans, meta, frames)
        self.log('AUC', auc_score, sync_dist=True)
        return auc_score


    def configure_optimizers(self) -> Dict:
        """
        Configure the optimizers and the learning rate schedulers.

        Returns:
            Dict: dictionary containing the optimizers, the learning rate schedulers and the metric to monitor
        """

        optimizer = Adam(self.parameters(), lr=self.learning_rate)
        scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99, last_epoch=-1)

        return {'optimizer': optimizer, 'lr_scheduler': scheduler, 'monitor': 'AUC'}


    def post_processing(self, out: np.ndarray, gt_data: np.ndarray, trans: np.ndarray, meta: np.ndarray, frames: np.ndarray) -> float:
        """
        Post processing of the model.

        Args:
            out (np.ndarray): output of the model
            gt_data (np.ndarray): ground truth data
            trans (np.ndarray): transformation index
            meta (np.ndarray): metadata
            frames (np.ndarray): frame indexes of the data

        Returns:
            float: auc score
        """

        all_gts = [file_name for file_name in os.listdir(self.gt_path) if file_name.endswith('.npy')]
        all_gts = sorted(all_gts)
        scene_clips = [(int(fn.split('_')[0]), int(fn.split('_')[1].split('.')[0])) for fn in all_gts]
        hr_ubnormal_masked_clips = get_hr_ubnormal_mask(self.split) if (self.use_hr and (self.dataset_name == 'UBnormal')) else {}
        hr_avenue_masked_clips = get_avenue_mask() if self.dataset_name == 'HR-Avenue' else {}

        num_transform = self.num_transforms
        model_scores_transf = {}
        dataset_gt_transf = {}

        for transformation in tqdm(range(num_transform)):
            # iterating over each transformation T

            dataset_gt = []
            model_scores = []
            cond_transform = (trans == transformation)

            out_transform, gt_data_transform, meta_transform, frames_transform = filter_vectors_by_cond([out, gt_data, meta, frames], cond_transform)

            for idx in range(len(all_gts)):
                # iterating over each clip C with transformation T

                scene_idx, clip_idx = scene_clips[idx]

                gt = np.load(os.path.join(self.gt_path, all_gts[idx]))
                n_frames = gt.shape[0]

                cond_scene_clip = (meta_transform[:, 0] == scene_idx) & (meta_transform[:, 1] == clip_idx)
                out_scene_clip, gt_scene_clip, meta_scene_clip, frames_scene_clip = filter_vectors_by_cond([out_transform, gt_data_transform,
                                                                                                            meta_transform, frames_transform],
                                                                                                           cond_scene_clip)

                figs_ids = sorted(list(set(meta_scene_clip[:, 2])))
                error_per_person = []
                error_per_person_max_loss = []

                for fig in figs_ids:
                    # iterating over each actor A in each clip C with transformation T

                    cond_fig = (meta_scene_clip[:, 2] == fig)
                    out_fig, _, frames_fig = filter_vectors_by_cond([out_scene_clip, gt_scene_clip, frames_scene_clip], cond_fig)
                    loss_matrix = compute_var_matrix(out_fig, frames_fig, n_frames)
                    fig_reconstruction_loss = np.nanmax(loss_matrix, axis=0)
                    if self.anomaly_score_pad_size != -1:
                        fig_reconstruction_loss = pad_scores(fig_reconstruction_loss, gt, self.anomaly_score_pad_size)

                    error_per_person.append(fig_reconstruction_loss)
                    error_per_person_max_loss.append(max(fig_reconstruction_loss))

                clip_score = np.stack(error_per_person, axis=0)
                clip_score_log = np.log1p(clip_score)
                clip_score = np.mean(clip_score, axis=0) + (np.amax(clip_score_log, axis=0)-np.amin(clip_score_log, axis=0))

                # removing the non-HR frames for UBnormal dataset
                if (scene_idx, clip_idx) in hr_ubnormal_masked_clips:
                    clip_score = clip_score[hr_ubnormal_masked_clips[(scene_idx, clip_idx)]]
                    gt = gt[hr_ubnormal_masked_clips[(scene_idx, clip_idx)]]

                # removing the non-HR frames for Avenue dataset
                if clip_idx in hr_avenue_masked_clips:
                    clip_score = clip_score[np.array(hr_avenue_masked_clips[clip_idx])==1]
                    gt = gt[np.array(hr_avenue_masked_clips[clip_idx])==1]

                # Abnormal score per frame
                clip_score = score_process(clip_score, self.anomaly_score_frames_shift, self.anomaly_score_filter_kernel_size)
                model_scores.append(clip_score)

                dataset_gt.append(gt)

            model_scores = np.concatenate(model_scores, axis=0)

            dataset_gt = np.concatenate(dataset_gt, axis=0)

            model_scores_transf[transformation] = model_scores
            dataset_gt_transf[transformation] = dataset_gt

        # aggregating the anomaly scores for all transformations
        pds = np.mean(np.stack(list(model_scores_transf.values()), 0), 0)
        gt = dataset_gt_transf[0]

        # computing the AUC
        auc = roc_auc_score(gt, pds)

        return auc


    def test_on_saved_tensors(self, split_name: str) -> float:
        """
        Skip the prediction step and test the model on the saved tensors.

        Args:
            split_name (str): split name (val, test)

        Returns:
            float: auc score
        """

        tensors = self._load_tensors(split_name, self.aggregation_strategy, self.n_generated_samples)
        auc_score = self.post_processing(tensors['prediction'], tensors['gt_data'], tensors['trans'],
                                         tensors['metadata'], tensors['frames'])
        print(f'AUC score: {auc_score:.6f}')
        return auc_score


    ## Helper functions

    def _aggregation_strategy(self, generated_xs: List[torch.Tensor], input_sequence: torch.Tensor, aggr_strategy: str) -> Tuple[torch.Tensor]:
        """
        Aggregates the generated samples and returns the selected one and its reconstruction error.
        Strategies:
            - all: returns all the generated samples
            - random: returns a random sample
            - best: returns the sample with the lowest reconstruction loss
            - worst: returns the sample with the highest reconstruction loss
            - mean: returns the mean of the losses of the generated samples
            - median: returns the median of the losses of the generated samples
            - mean_pose: returns the mean of the generated samples
            - median_pose: returns the median of the generated samples

        Args:
            generated_xs (List[torch.Tensor]): list of generated samples
            input_sequence (torch.Tensor): ground truth sequence
            aggr_strategy (str): aggregation strategy

        Raises:
            ValueError: if the aggregation strategy is not valid

        Returns:
            Tuple[torch.Tensor]: selected sample and its reconstruction error
        """

        aggr_strategy = self.aggregation_strategy if aggr_strategy is None else aggr_strategy
        if aggr_strategy == 'random':
            return generated_xs[np.random.randint(len(generated_xs))], None # Added None as it was missing

        B, repr_shape = input_sequence.shape[0], input_sequence.shape[1:]
        compute_loss = lambda x: torch.mean(self.loss_fn(x, input_sequence).reshape(-1, prod(repr_shape)), dim=-1)
        losses = [compute_loss(x) for x in generated_xs]

        if aggr_strategy == 'all':
            dims_idxs = list(range(2, len(repr_shape)+2))
            dims_idxs = [1, 0] + dims_idxs
            selected_x = torch.stack(generated_xs).permute(*dims_idxs)
            loss_of_selected_x = torch.stack(losses).permute(1, 0)
        elif aggr_strategy == 'mean':
            selected_x = None
            loss_of_selected_x = torch.mean(torch.stack(losses), dim=0)
        elif aggr_strategy == 'mean_pose':
            selected_x = torch.mean(torch.stack(generated_xs), dim=0)
            loss_of_selected_x = compute_loss(selected_x)
        elif aggr_strategy == 'median':
            loss_of_selected_x, _ = torch.median(torch.stack(losses), dim=0)
            selected_x = None
        elif aggr_strategy == 'median_pose':
            selected_x, _ = torch.median(torch.stack(generated_xs), dim=0)
            loss_of_selected_x = compute_loss(selected_x)
        elif aggr_strategy == 'best' or aggr_strategy == 'worst':
            strategy = (lambda x, y: x < y) if aggr_strategy == 'best' else (lambda x, y: x > y)
            loss_of_selected_x = torch.full((B,), fill_value=(1e10 if aggr_strategy == 'best' else -1.), device=self.device)
            selected_x = torch.zeros((B, *repr_shape)).to(self.device)

            for i in range(len(generated_xs)):
                mask = strategy(losses[i], loss_of_selected_x)
                loss_of_selected_x[mask] = losses[i][mask]
                selected_x[mask] = generated_xs[i][mask]
        elif 'quantile' in aggr_strategy:
            q = float(aggr_strategy.split(':')[-1])
            loss_of_selected_x = torch.quantile(torch.stack(losses), q, dim=0)
            selected_x = None
        else:
            raise ValueError(f'Unknown aggregation strategy {aggr_strategy}')

        # Ensuring selected_x and loss_of_selected_x are always returned
        if selected_x is None and loss_of_selected_x is None:
             # Default to mean loss if strategy doesn't return both
             loss_of_selected_x = torch.mean(torch.stack(losses), dim=0)


        return selected_x, loss_of_selected_x


    def _infer_number_of_joint(self, args: argparse.Namespace) -> int:
        """
        Infer the number of joints based on the dataset parameters.

        Args:
            args (argparse.Namespace): arguments containing the hyperparameters of the model

        Returns:
            int: number of joints
        """

        if args.headless:
            joints_to_consider = 14
        elif args.kp18_format:
            joints_to_consider = 18
        else:
            joints_to_consider = 17
        return joints_to_consider


    def _load_tensors(self, split_name: str, aggr_strategy: str, n_gen: int) -> Dict[str, torch.Tensor]:
        """
        Loads the tensors from the experiment directory.

        Args:
            split_name (str): name of the split (train, val, test)
            aggr_strategy (str): aggregation strategy
            n_gen (int): number of generated samples

        Returns:
            Dict[str, torch.Tensor]: dictionary containing the tensors. The keys are inferred from the file names.
        """

        name = 'saved_tensors_{}_{}_{}'.format(split_name, aggr_strategy, n_gen)
        path = os.path.join(self.ckpt_dir, name)
        if not os.path.exists(path):
            os.mkdir(path)
        tensor_files = os.listdir(path)
        tensors = {}
        for t_file in tensor_files:
            t_name = t_file.split('.')[0]
            tensors[t_name] = torch.load(os.path.join(path, t_file))
        return tensors


    def _pack_out_data(self, selected_x: torch.Tensor, loss_of_selected_x: torch.Tensor, additional_out: List[torch.Tensor], return_: str) -> List[torch.Tensor]:
        """
        Packs the output data according to the return_ argument.

        Args:
            selected_x (torch.Tensor): generated samples selected among the others according to the aggregation strategy
            loss_of_selected_x (torch.Tensor): loss of the selected samples
            additional_out (List[torch.Tensor]): additional output data (ground truth, meta-data, etc.)
            return_ (str): return strategy. Can be 'pose', 'loss', 'all'

        Raises:
            ValueError: if return_ is None and self.model_return_value is None

        Returns:
            List[torch.Tensor]: output data
        """

        if return_ is None:
            if self.model_return_value is None:
                raise ValueError('Either return_ or self.model_return_value must be set')
            else:
                return_ = self.model_return_value

        if return_ == 'poses':
            out = [selected_x]
        elif return_ == 'loss':
            out = [loss_of_selected_x]
        elif return_ == 'all':
            # Check if both are available before adding to the list
            out = []
            if loss_of_selected_x is not None:
                out.append(loss_of_selected_x)
            if selected_x is not None:
                 out.append(selected_x)

        return out + additional_out


    def _save_tensors(self, tensors: Dict[str, torch.Tensor], split_name: str, aggr_strategy: str, n_gen: int) -> None:
        """
        Saves the tensors in the experiment directory.

        Args:
            tensors (Dict[str, torch.Tensor]): tensors to save
            split_name (str): name of the split (val, test)
            aggr_strategy (str): aggregation strategy
            n_gen (int): number of generated samples
        """

        name = 'saved_tensors_{}_{}_{}'.format(split_name, aggr_strategy, n_gen)
        path = os.path.join(self.ckpt_dir, name)
        if not os.path.exists(path):
            os.mkdir(path)
        for t_name, tensor in tensors.items():
            torch.save(tensor, os.path.join(path, t_name + '.pt'))


    def _set_diffusion_variables(self) -> None:
        """
        Sets the diffusion variables.
        """

        self.noise_scheduler = Diffusion(noise_steps=self.noise_steps, n_joints=self.n_joints,
                                         device=self.device, time=self.n_frames)
        self._beta_ = self.noise_scheduler.schedule_noise()
        self._alpha_ = (1. - self._beta_)
        self._alpha_hat_ = torch.cumprod(self._alpha_, dim=0)

    def _unpack_data(self, x: torch.Tensor) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        """
        Unpacks the data.

        Args:
            x (torch.Tensor): list containing the input data, the transformation index, the metadata and the actual frames.

        Returns:
            Tuple[torch.Tensor, List[torch.Tensor]]: input data, list containing the transformation index, the metadata and the actual frames.
        """
        tensor_data = x[0].to(self.device)
        transformation_idx = x[1]
        metadata = x[2]
        actual_frames = x[3]
        meta_out = [transformation_idx, metadata, actual_frames]
        return tensor_data, meta_out


    @property
    def _beta(self) -> torch.Tensor:
        return self._beta_.to(self.device)


    @property
    def _alpha(self) -> torch.Tensor:
        return self._alpha_.to(self.device)


    @property
    def _alpha_hat(self) -> torch.Tensor:
        return self._alpha_hat_.to(self.device)

Overwriting /kaggle/working/DCMD-main/models/dcmd.py


In [39]:
%%writefile /kaggle/working/DCMD-main/eval_DCMD.py
import argparse
import os

import pytorch_lightning as pl
import yaml
from models.dcmd import DCMD
from utils.argparser import init_args
from utils.dataset import get_dataset_and_loader



if __name__== '__main__':
    
    # Parse command line arguments and load config file
    parser = argparse.ArgumentParser(description='DCMD')
    parser.add_argument('-c', '--config', type=str, required=True)
    args = parser.parse_args()
    args = yaml.load(open(args.config), Loader=yaml.FullLoader)
    args = argparse.Namespace(**args)
    args = init_args(args)

    # Initialize the model
    model = DCMD(args)
    
    if args.load_tensors:
        # Load tensors and test
        model.test_on_saved_tensors(split_name=args.split)
    else:
        # Load test data
        print('Loading data and creating loaders.....')
        ckpt_path = '/kaggle/input/checkpoints/kaggle/working/DCMD-main/checkpoints/HR-Avenue/train_experiment/last.ckpt'
        dataset, loader, _, _ = get_dataset_and_loader(args, split=args.split)
        
        # Initialize trainer and test
        trainer = pl.Trainer(accelerator=args.accelerator, devices=args.devices[:1],
                             default_root_dir=args.ckpt_dir, max_epochs=1, logger=False)
        out = trainer.test(model, dataloaders=loader, ckpt_path=ckpt_path)

Overwriting /kaggle/working/DCMD-main/eval_DCMD.py


In [40]:
%%writefile /kaggle/working/DCMD-main/utils/get_robust_data.py
import os
import numpy as np
import pickle

from copy import deepcopy

import torch

from utils.data import load_trajectories, extract_global_features
from utils.data import change_coordinate_system, scale_trajectories, aggregate_autoencoder_data
from utils.data import input_trajectories_missing_steps
from utils.preprocessing import remove_short_trajectories, aggregate_rnn_autoencoder_data


def save_scaler(scaler, path):
    with open(path, 'wb') as scaler_file:
        pickle.dump(scaler, scaler_file)
    
        
def load_scaler(path):
    with open(path, 'rb') as scaler_file:
        scaler = pickle.load(scaler_file)
    return scaler



# Load trajectory data and convert it into a format suitable for RNN autoencoders for training and testing of joint models
def data_of_combined_model(**args):
    # General
    exp_dir = args.get('exp_dir', '')
    split = args.get('split', 'train')
    normalize_pose = args.get('normalize_pose', False)
    trajectories_path = args.get('trajectories_path', '')
    include_global = args.get('include_global', True)
    debug = args.get('debug', False)
    if 'train' in split:
      # đổi thành path khác trong folder data nếu train dataset khác
        subfolder = '/kaggle/input/avenue/Avenue/training'
    elif 'test' in split:
        subfolder = '/kaggle/input/avenue/Avenue/testing'
    else:
        subfolder = 'validating'
    trajectories_path = os.path.join(trajectories_path, f'{subfolder}/trajectories')
    video_resolution = args.get('vid_res', (1080,720))
    video_resolution = np.array(video_resolution, dtype=np.float32)
    # Architecture
    reconstruct_original_data = args.get('reconstruct_original_data', False) 
    input_length = args.get('seg_len', 12)
    seg_stride = args.get('seg_stride', 1) - 1 
    pred_length = 0 
    # Training
    input_missing_steps = False # args.input_missing_steps
    
    if normalize_pose == True:
        global_normalisation_strategy = args.get('normalization_strategy', 'robust')
        local_normalisation_strategy = args.get('normalization_strategy', 'robust')
        out_normalisation_strategy = args.get('normalization_strategy', 'robust')


    trajectories = load_trajectories(trajectories_path, debug=debug, split=split)
    print('\nLoaded %d trajectories.' % len(trajectories))

    trajectories = remove_short_trajectories(trajectories, input_length=input_length,
                                             input_gap=seg_stride, pred_length=pred_length)
    print('\nRemoved short trajectories. Number of trajectories left: %d.' % len(trajectories))

    # trajectories, trajectories_val = split_into_train_and_test(trajectories, train_ratio=0.8, seed=42)

    if input_missing_steps:
        trajectories = input_trajectories_missing_steps(trajectories)
        print('\nInputted missing steps of trajectories.')

    # Global
    if include_global:
        global_trajectories = extract_global_features(deepcopy(trajectories), video_resolution=video_resolution)

        global_trajectories = change_coordinate_system(global_trajectories, video_resolution=video_resolution,
                                                        coordinate_system='global', invert=False)

        print('\nChanged global trajectories\'s coordinate system to global.')
        
        X_global, y_global, X_global_meta, y_global_meta = aggregate_rnn_autoencoder_data(global_trajectories, 
                                                                                        input_length=input_length,
                                                                                        input_gap=seg_stride, pred_length=pred_length, 
                                                                                        return_ids=True)
        
        if normalize_pose == True:
            # nếu test avenue thì dùng dòng này
            #/content/checkpoints/HR-Avenue/train_experiment/local_robust.pickle
            # default: scaler_path = os.path.join(exp_dir, f'global_{global_normalisation_strategy}.pickle')
            scaler_path = '/kaggle/input/checkpoints/kaggle/working/DCMD-main/checkpoints/HR-Avenue/train_experiment/local_robust.pickle'
            if split == 'train':
                _, global_scaler = scale_trajectories(aggregate_autoencoder_data(global_trajectories),
                                                    strategy=global_normalisation_strategy)
                save_scaler(global_scaler, scaler_path)
            else:
                global_scaler = load_scaler(scaler_path)

            X_global, _ = scale_trajectories(X_global, scaler=global_scaler, strategy=global_normalisation_strategy)
            
            if y_global is not None:
                y_global, _ = scale_trajectories(y_global, scaler=global_scaler,
                                                strategy=global_normalisation_strategy)
                
            print('\nNormalised global trajectories using the %s normalisation strategy.' % global_normalisation_strategy)
    
    else:
        X_global, X_global_meta = None, None
    
    # Local
    local_trajectories = deepcopy(trajectories) if reconstruct_original_data else trajectories

    local_trajectories = change_coordinate_system(local_trajectories, video_resolution=video_resolution,
                                                  coordinate_system='bounding_box_centre', invert=False)

    print('\nChanged local trajectories\'s coordinate system to bounding_box_centre.')

    X_local, y_local, X_local_meta, y_local_meta = aggregate_rnn_autoencoder_data(local_trajectories, input_length=input_length, 
                                                                                  input_gap=seg_stride, pred_length=pred_length,
                                                                                  return_ids=True)
    
    if normalize_pose == True:
        #scaler_path = '/content/drive/MyDrive/DCMD-main-main/checkpoints/Avenue/test_experiment/local_robust.pickle'
        scaler_path = os.path.join(exp_dir, f'local_{local_normalisation_strategy}.pickle')

        if split == 'train':
            _, local_scaler = scale_trajectories(aggregate_autoencoder_data(local_trajectories),
                                                strategy=local_normalisation_strategy)
            save_scaler(local_scaler, scaler_path)
        else:
            local_scaler = load_scaler("/kaggle/input/checkpoints/kaggle/working/DCMD-main/checkpoints/HR-Avenue/train_experiment/local_robust.pickle")

        X_local, _ = scale_trajectories(X_local, scaler=local_scaler, strategy=local_normalisation_strategy)

        if y_local is not None:
            y_local, _ = scale_trajectories(y_local, scaler=local_scaler, strategy=local_normalisation_strategy)
        
        print('\nNormalised local trajectories using the %s normalisation strategy.' % local_normalisation_strategy)

    # (Optional) Reconstruct the original data
    if reconstruct_original_data:
        print('\nReconstruction/Prediction target is the original data.')
        out_trajectories = trajectories
        
        out_trajectories = change_coordinate_system(out_trajectories, video_resolution=video_resolution,
                                                    coordinate_system='global', invert=False)
    
        print('\nChanged target trajectories\'s coordinate system to global.')
        
        scaler_path = os.path.join(exp_dir, f'out_{out_normalisation_strategy}.pickle')
    
        if split == 'train':
            _, out_scaler = scale_trajectories(aggregate_autoencoder_data(out_trajectories),
                                               strategy=out_normalisation_strategy)
            save_scaler(out_scaler, scaler_path)
        else:
            out_scaler = load_scaler(scaler_path)
        
        ######## X_out_{}, y_out_{} numpy arrays

        X_out, y_out, X_out_meta, y_out_meta = aggregate_rnn_autoencoder_data(out_trajectories, input_length=input_length, 
                                                                              input_gap=seg_stride, pred_length=pred_length,
                                                                              return_ids=True)

        X_out, _ = scale_trajectories(X_out, scaler=out_scaler, strategy=out_normalisation_strategy)
        
        if y_out is not None:
            y_out, _ = scale_trajectories(y_out, scaler=out_scaler, strategy=out_normalisation_strategy)
            
        print('\nNormalised target trajectories using the %s normalisation strategy.' % out_normalisation_strategy)
        
            
    if pred_length > 0:
        
        if reconstruct_original_data:
            return (X_global, X_global_meta), \
                   (X_local, X_local_meta), \
                   (X_out, X_out_meta), \
                   (y_global, y_global_meta), \
                   (y_local, y_local_meta), \
                   (y_out, y_out_meta) 
        else:
            return (X_global, X_global_meta), \
                   (X_local, X_local_meta), \
                   (y_global, y_global_meta), \
                   (y_local, y_local_meta)
    else:
        if reconstruct_original_data:
            return (X_global, X_global_meta), \
                   (X_local, X_local_meta), \
                   (X_out, X_out_meta)
        else:
            return (X_global, X_global_meta), \
                   (X_local, X_local_meta)

Overwriting /kaggle/working/DCMD-main/utils/get_robust_data.py


# **Train** 

In [41]:
!python /kaggle/working/DCMD-main/train_DCMD.py --config /kaggle/working/DCMD-main/config/Avenue/dcmd_train.yaml

Experiment directories created in /kaggle/working/DCMD-main/checkpoints/HR-Avenue/train_experiment

Loaded 2786 trajectories.

Removed short trajectories. Number of trajectories left: 1735.

Changed local trajectories's coordinate system to bounding_box_centre.

Normalised local trajectories using the robust normalisation strategy.
/usr/local/lib/python3.11/dist-packages/pytorch_lightning/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
/usr/local/lib/python3.11/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:751: Checkpoint directory /kaggle/working/DCMD-main/checkpoints/HR-Avenue/train_experiment exists and is not empty.
Epoch 9: 100%|████████████████████████████████| 185/185 [12:54<00:00,  0.24it/s]


If download checkpoint needed, run this code and download checkpoints.zip file in Output

In [42]:
# 1. Zip toàn bộ folder checkpoints
!zip -r /kaggle/working/checkpoints.zip /kaggle/working/DCMD-main/checkpoints

# 2. (tuỳ chọn) kiểm tra xem zip đã tạo xong chưa
!ls -lh /kaggle/working/checkpoints.zip


  adding: kaggle/working/DCMD-main/checkpoints/ (stored 0%)
  adding: kaggle/working/DCMD-main/checkpoints/HR-Avenue/ (stored 0%)
  adding: kaggle/working/DCMD-main/checkpoints/HR-Avenue/train_experiment/ (stored 0%)
  adding: kaggle/working/DCMD-main/checkpoints/HR-Avenue/train_experiment/config.yaml (deflated 63%)
  adding: kaggle/working/DCMD-main/checkpoints/HR-Avenue/train_experiment/local_robust.pickle (deflated 15%)
  adding: kaggle/working/DCMD-main/checkpoints/HR-Avenue/train_experiment/last.ckpt (deflated 7%)
-rw-r--r-- 1 root root 320M Oct  3 12:20 /kaggle/working/checkpoints.zip


# **Testing**

In [43]:
import os
os.makedirs('/kaggle/working/DCMD-main/checkpoints/HR-Avenue/test_experiment', exist_ok=True)

In [44]:
!python /kaggle/working/DCMD-main/eval_DCMD.py --config /kaggle/working/DCMD-main/config/Avenue/dcmd_test.yaml

Loading data and creating loaders.....

Loaded 2594 trajectories.

Removed short trajectories. Number of trajectories left: 1661.

Changed local trajectories's coordinate system to bounding_box_centre.
Traceback (most recent call last):
  File "/kaggle/working/DCMD-main/eval_DCMD.py", line 32, in <module>
    dataset, loader, _, _ = get_dataset_and_loader(args, split=args.split)
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/DCMD-main/utils/dataset.py", line 297, in get_dataset_and_loader
    dataset = PoseDatasetRobust(path_to_data=args.data_dir, 
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/DCMD-main/utils/dataset.py", line 212, in __init__
    super(PoseDatasetRobust, self).__init__(path_to_data,
  File "/kaggle/working/DCMD-main/utils/dataset.py", line 55, in __init__
    self.segs_data_np, self.segs_meta, self.person_keys, self.segs_ids = self.gen_dataset(ret_keys=True,
                     